# Notebook 4: Manual Curation and Final Export of the Character Corpus

This notebook represents the final stage of the character corpus pipeline. It takes the clustering results generated in Notebook 3 and provides the tools required to manually verify, refine, and export the final corpus.
The notebook consists of the following steps:

- **Generation of a Cluster Review Interface**
   Creates an interactive HTML interface that visualizes the generated DBSCAN clusters. The interface allows each cluster to be classified as:

   * **character-cluster**
   * **not-a-character-cluster**
   * **needs-manual-reworking**

- **Manual Cluster Curation**
   Reviews clusters requiring manual intervention. Mixed clusters can be split into multiple character clusters, merged with existing clusters, left unchanged, or rejected entirely. The resulting decisions are stored in a crop-level CSV that serves as the basis for the final corpus.

- **Duplicate Detection on Individual Pages**
   Performs a final quality-control step by identifying cases where multiple crops within the same character cluster originate from the same newspaper page. Such duplicates can be reviewed manually so that only the best crop is retained. (Characters only appear once per page, so two inside the same cluster is an error-source)

- **Final Corpus Export**
   Exports the curated corpus, including the finalized cluster assignments, page images, metadata, and summary reports into a structured directory suitable for further analysis and publication.

- **Creation of Visualisations**
   Creates two visualisations from the finalized corpus. The first one being a bar-chart, showing all character appearances per year over the whole publication span of 1871 until 1925. The second one showing the appearances of the five biggest-character clusters as a linechart over the publication span.

- **Creation of a simple HTML Frontend**
   Generates a lightweight HTML frontend that allows the exported corpus to be explored interactively. The frontend provides an overview of all character clusters and enables browsing of individual cluster contents and their corresponding source pages via IIIF-link.


##Environment
This notebook was created with the help of ChatGPT-5.5 and is supposed to be used in a Google Colab/Google Drive environment.

###1) Import Libraries

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import os
from urllib.parse import quote
from datetime import datetime
import re
import shutil
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError
from html import escape
import matplotlib.pyplot as plt

###2) Mounting Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

##3) Path Configurations

In [ ]:
# Root of the complete pipeline.
PIPELINE_ROOT = Path(
    "/content/drive/MyDrive/Masterarbeit_DH/"
    "Pipeline_building-character"
)

##4) Loading Clustering results and Creating a HTML-view

In [ ]:
# =====================================================
# Load clustering result for manual categorization
# =====================================================

# Select the clustering variant to inspect.
CLUSTERING_VARIANT_DIR = (
    PIPELINE_ROOT
    / "Embeddings"
    / "datacomp_vitl14_dbscan"
    / "clustering"
    / "dbscan_eps008_min3_cosine"
)

IMAGE_CLUSTERING_PATH = (
    CLUSTERING_VARIANT_DIR
    / "images.parquet"
)

CLUSTER_LEVEL_PATH = (
    CLUSTERING_VARIANT_DIR
    / "clusters.parquet"
)

CATEGORIZATION_HTML_PATH = (
    CLUSTERING_VARIANT_DIR
    / "cluster_categorization.html"
)


# -----------------------------------------------------
# Load clustering metadata
# -----------------------------------------------------

if not IMAGE_CLUSTERING_PATH.exists():
    raise FileNotFoundError(
        f"Clustering metadata not found:\n{IMAGE_CLUSTERING_PATH}"
    )

clustering_metadata = pd.read_parquet(
    IMAGE_CLUSTERING_PATH
)

if CLUSTER_LEVEL_PATH.exists():
    cluster_level_metadata = pd.read_parquet(
        CLUSTER_LEVEL_PATH
    )
else:
    cluster_level_metadata = (
        clustering_metadata.loc[
            clustering_metadata["cluster_id"] != -1
        ]
        .groupby("cluster_id")
        .size()
        .rename("cluster_size")
        .reset_index()
    )


required_columns = {
    "cluster_id",
    "embedding_index",
    "crop_id",
    "local_crop_path",
}

missing_columns = (
    required_columns
    - set(clustering_metadata.columns)
)

if missing_columns:
    raise ValueError(
        "The clustering metadata is missing required columns: "
        f"{sorted(missing_columns)}"
    )


# Exclude DBSCAN noise from cluster categorization.
clustered_images = clustering_metadata.loc[
    clustering_metadata["cluster_id"] != -1
].copy()

clustered_images["cluster_id"] = (
    clustered_images["cluster_id"].astype(int)
)

clustered_images = clustered_images.sort_values(
    [
        "cluster_id",
        "embedding_index",
    ]
).reset_index(drop=True)


print("=" * 70)
print("CLUSTERING RESULT LOADED")
print("=" * 70)
print("Variant directory:")
print(CLUSTERING_VARIANT_DIR)
print()
print(f"Images in clusters: {len(clustered_images):,}")
print(
    "Clusters:",
    f"{clustered_images['cluster_id'].nunique():,}",
)
print(
    "Smallest cluster:",
    f"{clustered_images.groupby('cluster_id').size().min():,}",
)
print(
    "Largest cluster:",
    f"{clustered_images.groupby('cluster_id').size().max():,}",
)
print("=" * 70)

display(clustered_images.head())

In [ ]:
# =====================================================
# Create interactive cluster categorization HTML
# =====================================================

# Maximum number of example images shown per cluster, use None to include every image.
MAX_IMAGES_PER_CLUSTER = 100

THUMBNAIL_SIZE = 180

CATEGORIES = [
    "not-a-character-cluster",
    "character-cluster",
    "needs-manual-reworking",
]


# -----------------------------------------------------
# Convert pipeline-relative paths into paths relative
# to the generated HTML file
# -----------------------------------------------------

def path_for_html(local_crop_path):
    absolute_image_path = (
        PIPELINE_ROOT
        / Path(str(local_crop_path))
    )

    relative_image_path = os.path.relpath(
        absolute_image_path,
        start=CATEGORIZATION_HTML_PATH.parent,
    )

    return quote(
        relative_image_path.replace("\\", "/"),
        safe="/:@?&=#%+-._~",
    )


# -----------------------------------------------------
# Build cluster records
# -----------------------------------------------------

cluster_records = []

grouped_clusters = clustered_images.groupby(
    "cluster_id",
    sort=True,
)

for cluster_id, cluster_rows in grouped_clusters:
    cluster_rows = cluster_rows.reset_index(drop=True)
    cluster_size = len(cluster_rows)

    if (
        MAX_IMAGES_PER_CLUSTER is not None
        and cluster_size > MAX_IMAGES_PER_CLUSTER
    ):
        # Select images evenly across the complete cluster instead
        # of taking only its first rows.
        selected_indices = np.linspace(
            0,
            cluster_size - 1,
            MAX_IMAGES_PER_CLUSTER,
            dtype=int,
        )

        displayed_rows = cluster_rows.iloc[
            selected_indices
        ]
    else:
        displayed_rows = cluster_rows

    images = []

    for row in displayed_rows.itertuples(index=False):
        local_path = str(row.local_crop_path)

        images.append(
            {
                "crop_id": str(row.crop_id),
                "embedding_index": int(
                    row.embedding_index
                ),
                "stored_path": local_path,
                "image_url": path_for_html(
                    local_path
                ),
            }
        )

    cluster_records.append(
        {
            "cluster_id": int(cluster_id),
            "cluster_size": int(cluster_size),
            "displayed_images": int(
                len(displayed_rows)
            ),
            "images": images,
        }
    )


# Display the largest clusters first.
cluster_records = sorted(
    cluster_records,
    key=lambda cluster: (
        -cluster["cluster_size"],
        cluster["cluster_id"],
    ),
)

cluster_data_json = json.dumps(
    cluster_records,
    ensure_ascii=False,
    separators=(",", ":"),
)

categories_json = json.dumps(
    CATEGORIES,
    ensure_ascii=False,
)

storage_key = (
    "cluster-categorization-"
    + CLUSTERING_VARIANT_DIR.name
)


# -----------------------------------------------------
# Build HTML
# -----------------------------------------------------

html_document = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">

    <meta
        name="viewport"
        content="width=device-width, initial-scale=1.0"
    >

    <title>Cluster Categorization</title>

    <style>
        * {{
            box-sizing: border-box;
        }}

        body {{
            margin: 0;
            font-family: Arial, Helvetica, sans-serif;
            background: #f3f4f6;
            color: #111827;
        }}

        header {{
            position: sticky;
            top: 0;
            z-index: 10;
            padding: 16px 24px;
            border-bottom: 1px solid #d1d5db;
            background: white;
        }}

        h1 {{
            margin: 0 0 8px;
        }}

        .progress-line {{
            display: flex;
            flex-wrap: wrap;
            gap: 18px;
            color: #4b5563;
        }}

        .toolbar {{
            display: flex;
            flex-wrap: wrap;
            gap: 10px;
            margin-top: 14px;
        }}

        button {{
            padding: 9px 14px;
            border: 1px solid #9ca3af;
            border-radius: 7px;
            background: white;
            cursor: pointer;
            font-weight: bold;
        }}

        button:hover {{
            background: #f3f4f6;
        }}

        button:disabled {{
            cursor: default;
            opacity: 0.45;
        }}

        main {{
            padding: 24px;
        }}

        .cluster-heading {{
            display: flex;
            flex-wrap: wrap;
            justify-content: space-between;
            gap: 12px;
            margin-bottom: 16px;
        }}

        .cluster-title {{
            font-size: 25px;
            font-weight: bold;
        }}

        .cluster-status {{
            padding: 8px 12px;
            border-radius: 7px;
            background: #e5e7eb;
            font-weight: bold;
        }}

        .category-controls {{
            display: flex;
            flex-wrap: wrap;
            gap: 10px;
            margin-bottom: 22px;
            padding: 16px;
            border: 1px solid #d1d5db;
            border-radius: 9px;
            background: white;
        }}

        .category-button {{
            flex: 1 1 240px;
            min-height: 45px;
        }}

        .category-button.selected {{
            border: 3px solid #111827;
            background: #dbeafe;
        }}

        .image-grid {{
            display: grid;
            grid-template-columns:
                repeat(
                    auto-fill,
                    minmax({THUMBNAIL_SIZE}px, 1fr)
                );
            gap: 14px;
        }}

        .image-card {{
            min-width: 0;
            padding: 9px;
            border: 1px solid #d1d5db;
            border-radius: 8px;
            background: white;
        }}

        .image-card img {{
            display: block;
            width: 100%;
            height: {THUMBNAIL_SIZE}px;
            object-fit: contain;
            border-radius: 5px;
            background: #f9fafb;
        }}

        .missing-image {{
            display: flex;
            align-items: center;
            justify-content: center;
            height: {THUMBNAIL_SIZE}px;
            border-radius: 5px;
            background: #e5e7eb;
            color: #6b7280;
            text-align: center;
        }}

        .crop-id {{
            margin-top: 7px;
            overflow-wrap: anywhere;
            font-size: 12px;
            font-weight: bold;
        }}

        .stored-path {{
            margin-top: 4px;
            overflow-wrap: anywhere;
            color: #6b7280;
            font-size: 10px;
        }}

        .sampling-note {{
            margin-bottom: 18px;
            color: #6b7280;
            font-style: italic;
        }}

        @media (max-width: 600px) {{
            main {{
                padding: 12px;
            }}

            header {{
                padding: 12px;
            }}

            .image-grid {{
                grid-template-columns:
                    repeat(2, minmax(0, 1fr));
            }}
        }}
    </style>
</head>

<body>
    <header>
        <h1>Cluster Categorization</h1>

        <div class="progress-line">
            <span id="cluster-position"></span>
            <span id="classification-progress"></span>
            <span id="category-counts"></span>
        </div>

        <div class="toolbar">
            <button id="previous-button">
                Previous cluster
            </button>

            <button id="next-button">
                Next cluster
            </button>

            <button id="next-unclassified-button">
                Next unclassified
            </button>

            <button id="export-button">
                Export classifications CSV
            </button>
        </div>
    </header>

    <main>
        <div class="cluster-heading">
            <div
                class="cluster-title"
                id="cluster-title"
            ></div>

            <div
                class="cluster-status"
                id="cluster-status"
            ></div>
        </div>

        <div
            class="category-controls"
            id="category-controls"
        ></div>

        <p
            class="sampling-note"
            id="sampling-note"
        ></p>

        <div
            class="image-grid"
            id="image-grid"
        ></div>
    </main>

    <script>
        const clusters = {cluster_data_json};
        const categories = {categories_json};
        const storageKey = {json.dumps(storage_key)};

        let currentPosition = 0;

        let classifications = JSON.parse(
            localStorage.getItem(storageKey) || "{{}}"
        );

        function escapeHtml(value) {{
            return String(value)
                .replaceAll("&", "&amp;")
                .replaceAll("<", "&lt;")
                .replaceAll(">", "&gt;")
                .replaceAll('"', "&quot;")
                .replaceAll("'", "&#039;");
        }}

        function saveClassifications() {{
            localStorage.setItem(
                storageKey,
                JSON.stringify(classifications)
            );
        }}

        function classificationFor(clusterId) {{
            return classifications[
                String(clusterId)
            ] || "";
        }}

        function classifyCurrentCluster(category) {{
            const cluster = clusters[currentPosition];

            classifications[
                String(cluster.cluster_id)
            ] = category;

            saveClassifications();
            renderCurrentCluster();
        }}

        function renderCategoryButtons(cluster) {{
            const selectedCategory =
                classificationFor(cluster.cluster_id);

            const controls = document.getElementById(
                "category-controls"
            );

            controls.innerHTML = categories
                .map(category => `
                    <button
                        class="category-button
                            ${{selectedCategory === category
                                ? "selected"
                                : ""}}"
                        data-category="${{escapeHtml(category)}}"
                    >
                        ${{escapeHtml(category)}}
                    </button>
                `)
                .join("");

            controls
                .querySelectorAll(".category-button")
                .forEach(button => {{
                    button.addEventListener(
                        "click",
                        () => classifyCurrentCluster(
                            button.dataset.category
                        )
                    );
                }});
        }}

        function renderImages(cluster) {{
            const grid = document.getElementById(
                "image-grid"
            );

            grid.innerHTML = cluster.images
                .map(image => `
                    <article class="image-card">
                        <a
                            href="${{escapeHtml(image.image_url)}}"
                            target="_blank"
                            rel="noopener"
                        >
                            <img
                                src="${{escapeHtml(image.image_url)}}"
                                alt="${{escapeHtml(image.crop_id)}}"
                                loading="lazy"
                                decoding="async"
                                onerror="
                                    this.closest('a').outerHTML =
                                    '<div class=&quot;missing-image&quot;>Image could not be loaded</div>'
                                "
                            >
                        </a>

                        <div class="crop-id">
                            ${{escapeHtml(image.crop_id)}}
                        </div>

                        <div class="stored-path">
                            ${{escapeHtml(image.stored_path)}}
                        </div>
                    </article>
                `)
                .join("");
        }}

        function updateProgress() {{
            const classifiedClusters = clusters.filter(
                cluster => classificationFor(
                    cluster.cluster_id
                )
            ).length;

            document.getElementById(
                "classification-progress"
            ).textContent =
                `Classified: ${{classifiedClusters}} / ` +
                `${{clusters.length}}`;

            const categoryCounts = categories.map(
                category => {{
                    const count = clusters.filter(
                        cluster =>
                            classificationFor(
                                cluster.cluster_id
                            ) === category
                    ).length;

                    return `${{category}}: ${{count}}`;
                }}
            );

            document.getElementById(
                "category-counts"
            ).textContent = categoryCounts.join(" | ");
        }}

        function renderCurrentCluster() {{
            const cluster = clusters[currentPosition];

            document.getElementById(
                "cluster-position"
            ).textContent =
                `Cluster ${{currentPosition + 1}} of ` +
                `${{clusters.length}}`;

            document.getElementById(
                "cluster-title"
            ).textContent =
                `Cluster ${{cluster.cluster_id}} — ` +
                `${{cluster.cluster_size.toLocaleString()}} images`;

            const category = classificationFor(
                cluster.cluster_id
            );

            document.getElementById(
                "cluster-status"
            ).textContent =
                category || "unclassified";

            const hiddenImages =
                cluster.cluster_size
                - cluster.displayed_images;

            document.getElementById(
                "sampling-note"
            ).textContent =
                hiddenImages > 0
                ? `${{cluster.displayed_images}} representative images ` +
                  `are shown; ${{hiddenImages.toLocaleString()}} ` +
                  `additional images are not displayed.`
                : `All ${{cluster.cluster_size.toLocaleString()}} ` +
                  `images are shown.`;

            renderCategoryButtons(cluster);
            renderImages(cluster);
            updateProgress();

            document.getElementById(
                "previous-button"
            ).disabled = currentPosition === 0;

            document.getElementById(
                "next-button"
            ).disabled =
                currentPosition === clusters.length - 1;

            window.scrollTo(0, 0);
        }}

        function moveToNextUnclassified() {{
            for (
                let offset = 1;
                offset <= clusters.length;
                offset++
            ) {{
                const candidatePosition =
                    (currentPosition + offset)
                    % clusters.length;

                const candidateCluster =
                    clusters[candidatePosition];

                if (
                    !classificationFor(
                        candidateCluster.cluster_id
                    )
                ) {{
                    currentPosition = candidatePosition;
                    renderCurrentCluster();
                    return;
                }}
            }}

            alert("All clusters have been classified.");
        }}

        function exportClassifications() {{
            const rows = [
                [
                    "cluster_id",
                    "cluster_size",
                    "cluster_category"
                ]
            ];

            clusters.forEach(cluster => {{
                rows.push([
                    cluster.cluster_id,
                    cluster.cluster_size,
                    classificationFor(
                        cluster.cluster_id
                    )
                ]);
            }});

            const csvText = rows
                .map(row =>
                    row
                        .map(value =>
                            '"' +
                            String(value)
                                .replaceAll('"', '""')
                            + '"'
                        )
                        .join(",")
                )
                .join("\\n");

            const blob = new Blob(
                [csvText],
                {{
                    type:
                        "text/csv;charset=utf-8"
                }}
            );

            const url = URL.createObjectURL(blob);
            const link = document.createElement("a");

            link.href = url;
            link.download =
                "cluster_classifications.csv";

            document.body.appendChild(link);
            link.click();
            link.remove();

            URL.revokeObjectURL(url);
        }}

        document
            .getElementById("previous-button")
            .addEventListener(
                "click",
                () => {{
                    if (currentPosition > 0) {{
                        currentPosition--;
                        renderCurrentCluster();
                    }}
                }}
            );

        document
            .getElementById("next-button")
            .addEventListener(
                "click",
                () => {{
                    if (
                        currentPosition
                        < clusters.length - 1
                    ) {{
                        currentPosition++;
                        renderCurrentCluster();
                    }}
                }}
            );

        document
            .getElementById("next-unclassified-button")
            .addEventListener(
                "click",
                moveToNextUnclassified
            );

        document
            .getElementById("export-button")
            .addEventListener(
                "click",
                exportClassifications
            );

        renderCurrentCluster();
    </script>
</body>
</html>
"""


# -----------------------------------------------------
# Save report
# -----------------------------------------------------

CATEGORIZATION_HTML_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

CATEGORIZATION_HTML_PATH.write_text(
    html_document,
    encoding="utf-8",
)

report_size_mb = (
    CATEGORIZATION_HTML_PATH.stat().st_size
    / 1024
    / 1024
)

print("=" * 70)
print("CLUSTER CATEGORIZATION REPORT CREATED")
print("=" * 70)
print(f"Clusters: {len(cluster_records):,}")
print(
    "Images included:",
    f"{sum(len(c['images']) for c in cluster_records):,}",
)
print(f"HTML size: {report_size_mb:.2f} MB")
print("Saved to:")
print(CATEGORIZATION_HTML_PATH)

##5) Manual Cluster-Curation Interface

 This interface reviews clusters that were previously classified as
 "needs-manual-reworking".

 For each such cluster, one of four final decisions can be made:

 1. Keep the original DBSCAN cluster unchanged.
 2. Create multiple independent final character clusters.
 3. Add the complete cluster to an existing DBSCAN character cluster.
 4. Reject the complete cluster as not-a-character-cluster.

 The exported CSV is a crop-level expansion of the original
 cluster_classifications.csv and is intended to become the final
 report used for exporting the character corpus.

In [ ]:
# =====================================================
# Create final manual cluster-curation interface
# =====================================================

# -----------------------------------------------------
# Paths
# -----------------------------------------------------

CLUSTER_CLASSIFICATIONS_PATH = (
    CLUSTERING_VARIANT_DIR
    / "cluster_classifications.csv"
)

FINAL_CLUSTER_REPORT_HTML_PATH = (
    CLUSTERING_VARIANT_DIR
    / "final_cluster_curation.html"
)


# -----------------------------------------------------
# Configuration
# -----------------------------------------------------

MANUAL_REWORK_CATEGORY = "needs-manual-reworking"

CHARACTER_CATEGORY = "character-cluster"

NOT_CHARACTER_CATEGORY = "not-a-character-cluster"

MANUAL_THUMBNAIL_SIZE = 180

# Initial number of independent final-cluster buttons shown
# when a DBSCAN cluster must be divided.
INITIAL_FINAL_CLUSTER_COUNT = 4


# -----------------------------------------------------
# Load cluster-level classifications
# -----------------------------------------------------

if not CLUSTER_CLASSIFICATIONS_PATH.exists():
    raise FileNotFoundError(
        "Cluster classifications were not found:\n"
        f"{CLUSTER_CLASSIFICATIONS_PATH}\n\n"
        "Upload or copy cluster_classifications.csv into "
        "the active clustering variant directory."
    )

cluster_classifications = pd.read_csv(
    CLUSTER_CLASSIFICATIONS_PATH
)

required_classification_columns = {
    "cluster_id",
    "cluster_category",
}

missing_classification_columns = (
    required_classification_columns
    - set(cluster_classifications.columns)
)

if missing_classification_columns:
    raise ValueError(
        "The classification CSV is missing columns: "
        f"{sorted(missing_classification_columns)}"
    )


# -----------------------------------------------------
# Normalize classification metadata
# -----------------------------------------------------

cluster_classifications = (
    cluster_classifications.copy()
)

cluster_classifications["cluster_id"] = (
    pd.to_numeric(
        cluster_classifications["cluster_id"],
        errors="raise",
    )
    .astype(int)
)

cluster_classifications["cluster_category"] = (
    cluster_classifications["cluster_category"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

if cluster_classifications[
    "cluster_id"
].duplicated().any():
    duplicated_cluster_ids = (
        cluster_classifications.loc[
            cluster_classifications[
                "cluster_id"
            ].duplicated(keep=False),
            "cluster_id",
        ]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    raise ValueError(
        "cluster_classifications.csv contains duplicate "
        "cluster_id values:\n"
        f"{duplicated_cluster_ids[:20]}"
    )


# -----------------------------------------------------
# Validate clustering metadata
# -----------------------------------------------------

required_clustering_columns = {
    "cluster_id",
    "crop_id",
    "embedding_index",
    "local_crop_path",
}

missing_clustering_columns = (
    required_clustering_columns
    - set(clustering_metadata.columns)
)

if missing_clustering_columns:
    raise ValueError(
        "The clustering metadata is missing columns: "
        f"{sorted(missing_clustering_columns)}"
    )

clustering_metadata = clustering_metadata.copy()

clustering_metadata["cluster_id"] = (
    pd.to_numeric(
        clustering_metadata["cluster_id"],
        errors="raise",
    )
    .astype(int)
)

clustering_metadata["crop_id"] = (
    clustering_metadata["crop_id"]
    .astype(str)
)

if clustering_metadata["crop_id"].duplicated().any():
    duplicated_crop_ids = (
        clustering_metadata.loc[
            clustering_metadata[
                "crop_id"
            ].duplicated(keep=False),
            "crop_id",
        ]
        .drop_duplicates()
        .tolist()
    )

    raise ValueError(
        "The clustering metadata contains duplicate "
        "crop_id values:\n"
        f"{duplicated_crop_ids[:20]}"
    )


# -----------------------------------------------------
# Select clusters requiring final manual decisions
# -----------------------------------------------------

manual_cluster_ids = (
    cluster_classifications.loc[
        cluster_classifications[
            "cluster_category"
        ]
        == MANUAL_REWORK_CATEGORY,
        "cluster_id",
    ]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

if not manual_cluster_ids:
    raise ValueError(
        "No clusters are classified as "
        f"'{MANUAL_REWORK_CATEGORY}'."
    )

manual_images = clustering_metadata.loc[
    clustering_metadata["cluster_id"].isin(
        manual_cluster_ids
    )
].copy()

manual_images = manual_images.sort_values(
    [
        "cluster_id",
        "embedding_index",
    ]
).reset_index(drop=True)

manual_clusters_without_images = sorted(
    set(manual_cluster_ids)
    - set(
        manual_images["cluster_id"]
        .drop_duplicates()
        .tolist()
    )
)

if manual_clusters_without_images:
    raise ValueError(
        "Some clusters classified as requiring manual "
        "reworking have no corresponding images in the "
        "clustering metadata:\n"
        f"{manual_clusters_without_images}"
    )


# -----------------------------------------------------
# Build lookup for possible merge targets
# -----------------------------------------------------

# Only clusters already classified as character-cluster are valid merge targets
cluster_category_lookup = dict(
    zip(
        cluster_classifications[
            "cluster_id"
        ].astype(int),
        cluster_classifications[
            "cluster_category"
        ].astype(str),
    )
)

cluster_category_lookup_json = json.dumps(
    cluster_category_lookup,
    ensure_ascii=False,
    separators=(",", ":"),
)


# -----------------------------------------------------
# Convert image paths for use inside the HTML
# -----------------------------------------------------

def manual_path_for_html(local_crop_path):
    local_crop_path = Path(
        str(local_crop_path)
    )

    if local_crop_path.is_absolute():
        absolute_image_path = (
            local_crop_path
        )
    else:
        absolute_image_path = (
            PIPELINE_ROOT
            / local_crop_path
        )

    relative_image_path = os.path.relpath(
        absolute_image_path,
        start=FINAL_CLUSTER_REPORT_HTML_PATH.parent,
    )

    return quote(
        relative_image_path.replace(
            "\\",
            "/",
        ),
        safe="/:@?&=#%+-._~",
    )


# -----------------------------------------------------
# Build records for the HTML interface
# -----------------------------------------------------

manual_cluster_records = []

for (
    cluster_id,
    cluster_rows,
) in manual_images.groupby(
    "cluster_id",
    sort=True,
):
    images = []

    for row in cluster_rows.itertuples(
        index=False
    ):
        local_path = str(
            row.local_crop_path
        )

        image_record = {
            "crop_id": str(
                row.crop_id
            ),
            "embedding_index": int(
                row.embedding_index
            ),
            "stored_path": local_path,
            "image_url": manual_path_for_html(
                local_path
            ),
        }

        # Preserve useful identifiers if they already exist
        # in the clustering metadata.
        for optional_column in [
            "page_id",
            "issue_id",
            "anno_id",
        ]:
            if hasattr(
                row,
                optional_column,
            ):
                value = getattr(
                    row,
                    optional_column,
                )

                if pd.isna(value):
                    value = ""

                image_record[
                    optional_column
                ] = str(value)

        images.append(
            image_record
        )

    manual_cluster_records.append(
        {
            "former_cluster_id": int(
                cluster_id
            ),
            "cluster_size": int(
                len(cluster_rows)
            ),
            "images": images,
        }
    )


# -----------------------------------------------------
# Build crop-level base report
# -----------------------------------------------------

# This merge copies all original classification columns onto every
# crop belonging to the corresponding DBSCAN cluster.

base_report = clustering_metadata.merge(
    cluster_classifications,
    on="cluster_id",
    how="left",
    validate="many_to_one",
)

base_report["cluster_category"] = (
    base_report["cluster_category"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)


# -----------------------------------------------------
# Add initial final-report columns
# -----------------------------------------------------

# former_cluster_id always records the original DBSCAN cluster for reproducability of the algorithmic and manual classification
base_report["former_cluster_id"] = (
    base_report["cluster_id"]
)

# Initial defaults:
#
# character-cluster:
#     accepted with its original DBSCAN cluster ID
#
# not-a-character-cluster:
#     excluded
#
# needs-manual-reworking:
#     pending a final manual decision
#
# blank or unspecified:
#     treated as excluded noise

base_report["final_decision"] = np.select(
    [
        base_report["cluster_category"]
        == CHARACTER_CATEGORY,

        base_report["cluster_category"]
        == NOT_CHARACTER_CATEGORY,

        base_report["cluster_category"]
        == MANUAL_REWORK_CATEGORY,
    ],
    [
        CHARACTER_CATEGORY,
        NOT_CHARACTER_CATEGORY,
        "pending-manual-decision",
    ],
    default=NOT_CHARACTER_CATEGORY,
)

base_report["final_cluster_id"] = np.where(
    base_report["final_decision"]
    == CHARACTER_CATEGORY,
    base_report["cluster_id"].astype(str),
    "",
)

base_report["cluster_creation_method"] = np.where(
    base_report["final_decision"]
    == CHARACTER_CATEGORY,
    "dbscan",
    "",
)

# Records used by JavaScript when constructing the final export.
base_report_records = (
    base_report
    .replace({np.nan: ""})
    .to_dict(orient="records")
)

manual_cluster_data_json = json.dumps(
    manual_cluster_records,
    ensure_ascii=False,
    separators=(",", ":"),
)

base_report_data_json = json.dumps(
    base_report_records,
    ensure_ascii=False,
    separators=(",", ":"),
)

base_report_columns_json = json.dumps(
    list(base_report.columns),
    ensure_ascii=False,
)

manual_storage_key = (
    "final-cluster-curation-"
    + CLUSTERING_VARIANT_DIR.name
)


# -----------------------------------------------------
# Build HTML
# -----------------------------------------------------

final_cluster_report_html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">

    <meta
        name="viewport"
        content="width=device-width, initial-scale=1.0"
    >

    <title>Final Cluster Curation</title>

    <style>
        * {{
            box-sizing: border-box;
        }}

        body {{
            margin: 0;
            font-family: Arial, Helvetica, sans-serif;
            background: #f3f4f6;
            color: #111827;
        }}

        header {{
            position: sticky;
            top: 0;
            z-index: 20;
            padding: 14px 22px;
            border-bottom: 1px solid #d1d5db;
            background: white;
        }}

        h1 {{
            margin: 0 0 8px;
        }}

        h2 {{
            margin-top: 0;
        }}

        .header-row,
        .decision-controls,
        .cluster-controls,
        .merge-controls {{
            display: flex;
            flex-wrap: wrap;
            align-items: center;
            gap: 10px;
        }}

        button {{
            padding: 9px 14px;
            border: 1px solid #9ca3af;
            border-radius: 6px;
            background: white;
            cursor: pointer;
            font-weight: 600;
        }}

        button:hover {{
            background: #f3f4f6;
        }}

        button:disabled {{
            opacity: 0.45;
            cursor: default;
        }}

        button.active {{
            outline: 4px solid #111827;
        }}

        #export-button {{
            background: #166534;
            color: white;
            border-color: #166534;
        }}

        #add-cluster-button {{
            background: #1d4ed8;
            color: white;
            border-color: #1d4ed8;
        }}

        .keep-button {{
            background: #dcfce7;
            border-color: #16a34a;
        }}

        .divide-button {{
            background: #dbeafe;
            border-color: #2563eb;
        }}

        .merge-button {{
            background: #fef3c7;
            border-color: #d97706;
        }}

        .reject-button {{
            background: #fee2e2;
            border-color: #dc2626;
        }}

        main {{
            padding: 22px;
        }}

        .cluster-summary,
        #merge-interface {{
            margin-bottom: 16px;
            padding: 14px;
            border: 1px solid #d1d5db;
            border-radius: 8px;
            background: white;
        }}

        .decision-controls {{
            margin: 16px 0;
        }}

        .instructions {{
            margin-top: 12px;
            color: #4b5563;
        }}

        .cluster-controls {{
            margin: 16px 0;
        }}

        .final-cluster-button {{
            min-width: 145px;
            margin-right: 8px;
            margin-bottom: 8px;
        }}

        .exclude-button {{
            background: #e5e7eb;
        }}

        #merge-target-input {{
            width: 220px;
            padding: 9px;
            border: 1px solid #9ca3af;
            border-radius: 6px;
            font-size: 14px;
        }}

        #save-merge-target-button {{
            background: #d97706;
            border-color: #d97706;
            color: white;
        }}

        .image-grid {{
            display: grid;
            grid-template-columns:
                repeat(
                    auto-fill,
                    minmax(
                        {MANUAL_THUMBNAIL_SIZE + 30}px,
                        1fr
                    )
                );
            gap: 14px;
        }}

        .image-card {{
            position: relative;
            padding: 8px;
            border: 4px solid #d1d5db;
            border-radius: 8px;
            background: white;
        }}

        .image-card.clickable {{
            cursor: pointer;
        }}

        .image-card.clickable:hover {{
            transform: translateY(-1px);
        }}

        .image-card img {{
            display: block;
            width: 100%;
            height: {MANUAL_THUMBNAIL_SIZE}px;
            object-fit: contain;
            background: #f9fafb;
        }}

        .image-label {{
            margin-top: 7px;
            overflow-wrap: anywhere;
            font-size: 12px;
        }}

        .assignment-badge {{
            position: absolute;
            top: 12px;
            right: 12px;
            max-width: calc(100% - 24px);
            padding: 5px 7px;
            border-radius: 999px;
            background: #111827;
            color: white;
            overflow: hidden;
            text-overflow: ellipsis;
            white-space: nowrap;
            font-size: 11px;
            font-weight: bold;
        }}

        .assignment-0 {{
            border-color: #d1d5db;
        }}

        .assignment-1 {{
            border-color: #dc2626;
        }}

        .assignment-2 {{
            border-color: #2563eb;
        }}

        .assignment-3 {{
            border-color: #16a34a;
        }}

        .assignment-4 {{
            border-color: #9333ea;
        }}

        .assignment-5 {{
            border-color: #ea580c;
        }}

        .assignment-6 {{
            border-color: #0891b2;
        }}

        .assignment-7 {{
            border-color: #be123c;
        }}

        .assignment-8 {{
            border-color: #4f46e5;
        }}

        .assignment-9 {{
            border-color: #65a30d;
        }}

        .assignment-10 {{
            border-color: #a16207;
        }}

        .status-line {{
            margin-top: 8px;
            font-weight: 600;
        }}

        .warning {{
            color: #b91c1c;
            font-weight: 700;
        }}

        .success {{
            color: #166534;
            font-weight: 700;
        }}

        .hidden {{
            display: none;
        }}
    </style>
</head>

<body>
    <header>
        <h1>Final Cluster Curation</h1>

        <div class="header-row">
            <button id="previous-button">
                Previous cluster
            </button>

            <button id="next-button">
                Next cluster
            </button>

            <button id="export-button">
                Export final cluster report
            </button>
        </div>

        <div
            id="header-progress"
            class="status-line"
        ></div>
    </header>

    <main>
        <div class="cluster-summary">
            <h2 id="cluster-title"></h2>

            <div id="cluster-status"></div>

            <div class="decision-controls">
                <button
                    id="keep-button"
                    class="keep-button"
                >
                    Keep as one character cluster
                </button>

                <button
                    id="divide-button"
                    class="divide-button"
                >
                    Create separate final clusters
                </button>

                <button
                    id="merge-button"
                    class="merge-button"
                >
                    Add to existing DBSCAN cluster
                </button>

                <button
                    id="reject-button"
                    class="reject-button"
                >
                    Not a character cluster
                </button>
            </div>

            <div
                id="decision-instructions"
                class="instructions"
            ></div>
        </div>


        <div
            id="merge-interface"
            class="hidden"
        >
            <div class="merge-controls">
                <label for="merge-target-input">
                    Existing DBSCAN cluster ID:
                </label>

                <input
                    id="merge-target-input"
                    type="number"
                    step="1"
                    placeholder="For example: 27"
                >

                <button id="save-merge-target-button">
                    Save target cluster
                </button>
            </div>

            <div
                id="merge-validation-message"
                class="status-line"
            ></div>
        </div>


        <div
            id="division-interface"
            class="hidden"
        >
            <div class="cluster-controls">
                <button
                    id="exclude-button"
                    class="final-cluster-button exclude-button"
                >
                    Exclude crop
                </button>

                <span id="final-cluster-buttons"></span>

                <button id="add-cluster-button">
                    Add final cluster
                </button>
            </div>
        </div>


        <div
            id="image-grid"
            class="image-grid"
        ></div>
    </main>


    <script>
        const clusters =
            {manual_cluster_data_json};

        const baseReport =
            {base_report_data_json};

        const originalColumns =
            {base_report_columns_json};

        const clusterCategoryLookup =
            {cluster_category_lookup_json};

        const storageKey =
            {json.dumps(manual_storage_key)};

        const manualReworkCategory =
            {json.dumps(MANUAL_REWORK_CATEGORY)};

        const characterCategory =
            {json.dumps(CHARACTER_CATEGORY)};

        const notCharacterCategory =
            {json.dumps(NOT_CHARACTER_CATEGORY)};


        // -------------------------------------------------
        // Load or initialize saved browser state
        // -------------------------------------------------

        let savedState =
            JSON.parse(
                localStorage.getItem(storageKey)
                || "{{}}"
            );

        if (!savedState.decisions) {{
            savedState.decisions = {{}};
        }}

        if (!savedState.assignments) {{
            savedState.assignments = {{}};
        }}

        if (!savedState.clusterCounts) {{
            savedState.clusterCounts = {{}};
        }}

        if (!savedState.mergeTargets) {{
            savedState.mergeTargets = {{}};
        }}

        let currentClusterPosition = 0;

        // 0 means exclude crop during manual division.
        let activeFinalClusterNumber = 1;


        // -------------------------------------------------
        // Browser-state helpers
        // -------------------------------------------------

        function saveState() {{
            localStorage.setItem(
                storageKey,
                JSON.stringify(savedState)
            );
        }}


        function formerClusterKey(
            formerClusterId
        ) {{
            return String(
                formerClusterId
            );
        }}


        function assignmentKey(
            formerClusterId,
            cropId
        ) {{
            return (
                String(formerClusterId)
                + "::"
                + String(cropId)
            );
        }}


        function getDecision(
            formerClusterId
        ) {{
            return (
                savedState.decisions[
                    formerClusterKey(
                        formerClusterId
                    )
                ]
                || ""
            );
        }}


        function setDecision(
            formerClusterId,
            decision
        ) {{
            const key =
                formerClusterKey(
                    formerClusterId
                );

            savedState.decisions[key] =
                decision;

            if (
                decision
                === "create-separate-final-clusters"
                && !savedState.clusterCounts[key]
            ) {{
                savedState.clusterCounts[key] =
                    {INITIAL_FINAL_CLUSTER_COUNT};
            }}

            saveState();
            renderCurrentCluster();
        }}


        function getFinalClusterCount(
            formerClusterId
        ) {{
            const key =
                formerClusterKey(
                    formerClusterId
                );

            return Number(
                savedState.clusterCounts[key]
                || {INITIAL_FINAL_CLUSTER_COUNT}
            );
        }}


        function setFinalClusterCount(
            formerClusterId,
            count
        ) {{
            savedState.clusterCounts[
                formerClusterKey(
                    formerClusterId
                )
            ] = count;

            saveState();
        }}


        function getAssignment(
            formerClusterId,
            cropId
        ) {{
            const key = assignmentKey(
                formerClusterId,
                cropId
            );

            if (
                savedState.assignments[key]
                === undefined
            ) {{
                return null;
            }}

            return Number(
                savedState.assignments[key]
            );
        }}


        function setAssignment(
            formerClusterId,
            cropId,
            assignment
        ) {{
            const key = assignmentKey(
                formerClusterId,
                cropId
            );

            savedState.assignments[key] =
                Number(assignment);

            saveState();
        }}


        function getMergeTarget(
            formerClusterId
        ) {{
            const key =
                formerClusterKey(
                    formerClusterId
                );

            if (
                savedState.mergeTargets[key]
                === undefined
            ) {{
                return "";
            }}

            return String(
                savedState.mergeTargets[key]
            );
        }}


        function setMergeTarget(
            formerClusterId,
            targetClusterId
        ) {{
            savedState.mergeTargets[
                formerClusterKey(
                    formerClusterId
                )
            ] = String(
                targetClusterId
            );

            saveState();
        }}


        // -------------------------------------------------
        // Final cluster ID helper
        // -------------------------------------------------

        function provisionalFinalClusterId(
            formerClusterId,
            assignmentNumber
        ) {{
            return (
                "manual_"
                + String(formerClusterId)
                + "_"
                + String(assignmentNumber)
                    .padStart(2, "0")
            );
        }}


        // -------------------------------------------------
        // Merge validation
        // -------------------------------------------------

        function validateMergeTarget(
            formerClusterId,
            targetValue
        ) {{
            const targetText =
                String(
                    targetValue
                    ?? ""
                ).trim();

            if (targetText === "") {{
                return {{
                    valid: false,
                    message:
                        "Enter an existing DBSCAN cluster ID."
                }};
            }}

            const targetClusterId =
                Number(targetText);

            if (
                !Number.isInteger(
                    targetClusterId
                )
            ) {{
                return {{
                    valid: false,
                    message:
                        "The target cluster ID must be an integer."
                }};
            }}

            if (
                targetClusterId
                === Number(formerClusterId)
            ) {{
                return {{
                    valid: false,
                    message:
                        "A cluster cannot be added to itself."
                }};
            }}

            const targetCategory =
                clusterCategoryLookup[
                    String(targetClusterId)
                ];

            if (
                targetCategory === undefined
            ) {{
                return {{
                    valid: false,
                    message:
                        "This DBSCAN cluster ID does not exist "
                        + "in cluster_classifications.csv."
                }};
            }}

            if (
                targetCategory
                === notCharacterCategory
            ) {{
                return {{
                    valid: false,
                    message:
                        "The selected target is classified as "
                        + "not-a-character-cluster."
                }};
            }}

            if (
                targetCategory
                === manualReworkCategory
            ) {{
                return {{
                    valid: false,
                    message:
                        "The selected target still requires "
                        + "manual reworking and cannot yet be "
                        + "used as a stable target cluster."
                }};
            }}

            if (
                targetCategory
                !== characterCategory
            ) {{
                return {{
                    valid: false,
                    message:
                        "The selected target is not a confirmed "
                        + "character-cluster."
                }};
            }}

            return {{
                valid: true,
                targetClusterId:
                    targetClusterId,
                message:
                    "This cluster will be added to existing "
                    + "DBSCAN cluster "
                    + targetClusterId
                    + "."
            }};
        }}


        // -------------------------------------------------
        // Decision-button rendering
        // -------------------------------------------------

        function renderDecisionButtons(
            decision
        ) {{
            document.getElementById(
                "keep-button"
            ).classList.toggle(
                "active",
                decision
                === "keep-as-character-cluster"
            );

            document.getElementById(
                "divide-button"
            ).classList.toggle(
                "active",
                decision
                === "create-separate-final-clusters"
            );

            document.getElementById(
                "merge-button"
            ).classList.toggle(
                "active",
                decision
                === "add-to-existing-cluster"
            );

            document.getElementById(
                "reject-button"
            ).classList.toggle(
                "active",
                decision
                === "not-a-character-cluster"
            );
        }}


        // -------------------------------------------------
        // Division-interface rendering
        // -------------------------------------------------

        function renderFinalClusterControls(
            cluster,
            decision
        ) {{
            const divisionInterface =
                document.getElementById(
                    "division-interface"
                );

            if (
                decision
                !== "create-separate-final-clusters"
            ) {{
                divisionInterface.classList.add(
                    "hidden"
                );

                return;
            }}

            divisionInterface.classList.remove(
                "hidden"
            );

            const container =
                document.getElementById(
                    "final-cluster-buttons"
                );

            container.innerHTML = "";

            const excludeButton =
                document.getElementById(
                    "exclude-button"
                );

            excludeButton.classList.toggle(
                "active",
                activeFinalClusterNumber === 0
            );

            const count =
                getFinalClusterCount(
                    cluster.former_cluster_id
                );

            for (
                let number = 1;
                number <= count;
                number++
            ) {{
                const button =
                    document.createElement(
                        "button"
                    );

                button.textContent =
                    provisionalFinalClusterId(
                        cluster.former_cluster_id,
                        number
                    );

                button.className =
                    "final-cluster-button "
                    + "assignment-"
                    + number;

                if (
                    activeFinalClusterNumber
                    === number
                ) {{
                    button.classList.add(
                        "active"
                    );
                }}

                button.addEventListener(
                    "click",
                    () => {{
                        activeFinalClusterNumber =
                            number;

                        renderCurrentCluster();
                    }}
                );

                container.appendChild(
                    button
                );
            }}
        }}


        // -------------------------------------------------
        // Merge-interface rendering
        // -------------------------------------------------

        function renderMergeInterface(
            cluster,
            decision
        ) {{
            const mergeInterface =
                document.getElementById(
                    "merge-interface"
                );

            if (
                decision
                !== "add-to-existing-cluster"
            ) {{
                mergeInterface.classList.add(
                    "hidden"
                );

                return;
            }}

            mergeInterface.classList.remove(
                "hidden"
            );

            const input =
                document.getElementById(
                    "merge-target-input"
                );

            const savedTarget =
                getMergeTarget(
                    cluster.former_cluster_id
                );

            input.value =
                savedTarget;

            const message =
                document.getElementById(
                    "merge-validation-message"
                );

            if (savedTarget === "") {{
                message.textContent =
                    "Enter the existing DBSCAN character "
                    + "cluster to which this complete cluster "
                    + "should be added.";

                message.classList.remove(
                    "warning",
                    "success"
                );

                return;
            }}

            const validation =
                validateMergeTarget(
                    cluster.former_cluster_id,
                    savedTarget
                );

            message.textContent =
                validation.message;

            message.classList.toggle(
                "warning",
                !validation.valid
            );

            message.classList.toggle(
                "success",
                validation.valid
            );
        }}


        // -------------------------------------------------
        // Image rendering
        // -------------------------------------------------

        function renderImages(
            cluster,
            decision
        ) {{
            const grid =
                document.getElementById(
                    "image-grid"
                );

            grid.innerHTML = "";

            cluster.images.forEach(
                image => {{
                    let assignment = null;

                    if (
                        decision
                        === "create-separate-final-clusters"
                    ) {{
                        assignment =
                            getAssignment(
                                cluster.former_cluster_id,
                                image.crop_id
                            );
                    }}

                    const visualAssignment =
                        assignment === null
                        ? 0
                        : assignment;

                    const card =
                        document.createElement(
                            "div"
                        );

                    card.className =
                        "image-card assignment-"
                        + visualAssignment;

                    if (
                        decision
                        === "create-separate-final-clusters"
                    ) {{
                        card.classList.add(
                            "clickable"
                        );
                    }}

                    const imageElement =
                        document.createElement(
                            "img"
                        );

                    imageElement.src =
                        image.image_url;

                    imageElement.alt =
                        image.crop_id;

                    imageElement.loading =
                        "lazy";

                    const badge =
                        document.createElement(
                            "div"
                        );

                    badge.className =
                        "assignment-badge";

                    if (
                        decision
                        === "keep-as-character-cluster"
                    ) {{
                        badge.textContent =
                            "Keep unchanged";
                    }} else if (
                        decision
                        === "not-a-character-cluster"
                    ) {{
                        badge.textContent =
                            "Excluded";
                    }} else if (
                        decision
                        === "add-to-existing-cluster"
                    ) {{
                        const target =
                            getMergeTarget(
                                cluster.former_cluster_id
                            );

                        badge.textContent =
                            target === ""
                            ? "Target required"
                            : "Add to cluster "
                                + target;
                    }} else if (
                        decision
                        === "create-separate-final-clusters"
                    ) {{
                        if (
                            assignment === null
                        ) {{
                            badge.textContent =
                                "Unassigned";
                        }} else if (
                            assignment === 0
                        ) {{
                            badge.textContent =
                                "Excluded";
                        }} else {{
                            badge.textContent =
                                provisionalFinalClusterId(
                                    cluster.former_cluster_id,
                                    assignment
                                );
                        }}
                    }} else {{
                        badge.textContent =
                            "Decision required";
                    }}

                    const label =
                        document.createElement(
                            "div"
                        );

                    label.className =
                        "image-label";

                    label.textContent =
                        image.crop_id;

                    card.appendChild(
                        imageElement
                    );

                    card.appendChild(
                        badge
                    );

                    card.appendChild(
                        label
                    );

                    if (
                        decision
                        === "create-separate-final-clusters"
                    ) {{
                        card.addEventListener(
                            "click",
                            () => {{
                                setAssignment(
                                    cluster.former_cluster_id,
                                    image.crop_id,
                                    activeFinalClusterNumber
                                );

                                renderCurrentCluster();
                            }}
                        );
                    }}

                    grid.appendChild(
                        card
                    );
                }}
            );
        }}


        // -------------------------------------------------
        // Completion-state helper
        // -------------------------------------------------

        function isClusterComplete(
            cluster
        ) {{
            const decision =
                getDecision(
                    cluster.former_cluster_id
                );

            if (
                decision
                === "keep-as-character-cluster"
                || decision
                === "not-a-character-cluster"
            ) {{
                return true;
            }}

            if (
                decision
                === "add-to-existing-cluster"
            ) {{
                const validation =
                    validateMergeTarget(
                        cluster.former_cluster_id,
                        getMergeTarget(
                            cluster.former_cluster_id
                        )
                    );

                return validation.valid;
            }}

            if (
                decision
                === "create-separate-final-clusters"
            ) {{
                return cluster.images.every(
                    image =>
                        getAssignment(
                            cluster.former_cluster_id,
                            image.crop_id
                        ) !== null
                );
            }}

            return false;
        }}


        // -------------------------------------------------
        // Status rendering
        // -------------------------------------------------

        function updateStatus(
            cluster,
            decision
        ) {{
            let statusText = "";

            if (
                decision
                === "keep-as-character-cluster"
            ) {{
                statusText =
                    "This DBSCAN cluster will remain one "
                    + "final character cluster.";
            }} else if (
                decision
                === "not-a-character-cluster"
            ) {{
                statusText =
                    "All crops in this DBSCAN cluster "
                    + "will be excluded from the corpus.";
            }} else if (
                decision
                === "add-to-existing-cluster"
            ) {{
                const validation =
                    validateMergeTarget(
                        cluster.former_cluster_id,
                        getMergeTarget(
                            cluster.former_cluster_id
                        )
                    );

                statusText =
                    validation.message;
            }} else if (
                decision
                === "create-separate-final-clusters"
            ) {{
                const reviewedCount =
                    cluster.images.filter(
                        image =>
                            getAssignment(
                                cluster.former_cluster_id,
                                image.crop_id
                            ) !== null
                    ).length;

                const includedCount =
                    cluster.images.filter(
                        image => {{
                            const assignment =
                                getAssignment(
                                    cluster.former_cluster_id,
                                    image.crop_id
                                );

                            return (
                                assignment !== null
                                && assignment > 0
                            );
                        }}
                    ).length;

                const excludedCount =
                    cluster.images.filter(
                        image =>
                            getAssignment(
                                cluster.former_cluster_id,
                                image.crop_id
                            ) === 0
                    ).length;

                statusText =
                    reviewedCount.toLocaleString()
                    + " of "
                    + cluster.cluster_size.toLocaleString()
                    + " crops reviewed | Included: "
                    + includedCount.toLocaleString()
                    + " | Excluded: "
                    + excludedCount.toLocaleString();
            }} else {{
                statusText =
                    "Choose a final decision for this cluster.";
            }}

            document.getElementById(
                "cluster-status"
            ).textContent =
                statusText;

            const completedClusters =
                clusters.filter(
                    candidate =>
                        isClusterComplete(
                            candidate
                        )
                ).length;

            document.getElementById(
                "header-progress"
            ).textContent =
                "Cluster "
                + (
                    currentClusterPosition + 1
                )
                + " of "
                + clusters.length
                + " | Completed: "
                + completedClusters
                + " of "
                + clusters.length;
        }}


        // -------------------------------------------------
        // Main rendering function
        // -------------------------------------------------

        function renderCurrentCluster() {{
            const cluster =
                clusters[
                    currentClusterPosition
                ];

            const decision =
                getDecision(
                    cluster.former_cluster_id
                );

            document.getElementById(
                "cluster-title"
            ).textContent =
                "Former DBSCAN cluster "
                + cluster.former_cluster_id
                + " — "
                + cluster.cluster_size.toLocaleString()
                + " crops";

            document.getElementById(
                "previous-button"
            ).disabled =
                currentClusterPosition === 0;

            document.getElementById(
                "next-button"
            ).disabled =
                currentClusterPosition
                === clusters.length - 1;

            renderDecisionButtons(
                decision
            );

            renderFinalClusterControls(
                cluster,
                decision
            );

            renderMergeInterface(
                cluster,
                decision
            );

            renderImages(
                cluster,
                decision
            );

            updateStatus(
                cluster,
                decision
            );

            const instructions =
                document.getElementById(
                    "decision-instructions"
                );

            if (
                decision
                === "create-separate-final-clusters"
            ) {{
                instructions.textContent =
                    "Select a final-cluster button and "
                    + "click the crops belonging to that "
                    + "independent character cluster. "
                    + "Use “Exclude crop” for advertisements "
                    + "or other material that should not enter "
                    + "the character corpus.";
            }} else if (
                decision
                === "add-to-existing-cluster"
            ) {{
                instructions.textContent =
                    "Enter the ID of the existing confirmed "
                    + "DBSCAN character cluster to which all "
                    + "crops in the current cluster belong.";
            }} else {{
                instructions.textContent =
                    "Choose whether this DBSCAN cluster should "
                    + "remain unchanged, be divided into "
                    + "independent final clusters, be added to "
                    + "an existing cluster, or be rejected.";
            }}

            window.scrollTo(
                0,
                0
            );
        }}


        // -------------------------------------------------
        // CSV helper
        // -------------------------------------------------

        function csvEscape(
            value
        ) {{
            if (
                value === null
                || value === undefined
            ) {{
                value = "";
            }}

            return (
                '"'
                + String(value)
                    .replaceAll(
                        '"',
                        '""'
                    )
                + '"'
            );
        }}


        // -------------------------------------------------
        // Final report construction and export
        // -------------------------------------------------

        function exportFinalReport() {{
            const incompleteClusters =
                clusters.filter(
                    cluster =>
                        !isClusterComplete(
                            cluster
                        )
                );

            if (
                incompleteClusters.length > 0
            ) {{
                const incompleteIds =
                    incompleteClusters
                    .map(
                        cluster =>
                            cluster.former_cluster_id
                    )
                    .slice(
                        0,
                        20
                    )
                    .join(", ");

                const continueExport =
                    window.confirm(
                        incompleteClusters.length
                        + " cluster(s) still have no complete "
                        + "final decision. Examples: "
                        + incompleteIds
                        + "\\n\\n"
                        + "These rows will remain marked as "
                        + "pending-manual-decision. Export anyway?"
                    );

                if (!continueExport) {{
                    return;
                }}
            }}


            const finalRows =
                baseReport.map(
                    row => {{
                        const output = {{
                            ...row
                        }};

                        const formerClusterId =
                            Number(
                                output.cluster_id
                            );

                        const originalCategory =
                            String(
                                output.cluster_category
                                ?? ""
                            );

                        // Only clusters originally marked for
                        // manual reworking need browser decisions.
                        if (
                            originalCategory
                            === manualReworkCategory
                        ) {{
                            const decision =
                                getDecision(
                                    formerClusterId
                                );

                            if (
                                decision
                                === "keep-as-character-cluster"
                            ) {{
                                output.final_decision =
                                    characterCategory;

                                output.final_cluster_id =
                                    String(
                                        formerClusterId
                                    );

                                output.former_cluster_id =
                                    String(
                                        formerClusterId
                                    );

                                output.cluster_creation_method =
                                    "dbscan_manual_confirmed";
                            }} else if (
                                decision
                                === "not-a-character-cluster"
                            ) {{
                                output.final_decision =
                                    notCharacterCategory;

                                output.final_cluster_id =
                                    "";

                                output.former_cluster_id =
                                    String(
                                        formerClusterId
                                    );

                                output.cluster_creation_method =
                                    "";
                            }} else if (
                                decision
                                === "add-to-existing-cluster"
                            ) {{
                                const target =
                                    getMergeTarget(
                                        formerClusterId
                                    );

                                const validation =
                                    validateMergeTarget(
                                        formerClusterId,
                                        target
                                    );

                                if (
                                    validation.valid
                                ) {{
                                    output.final_decision =
                                        characterCategory;

                                    output.final_cluster_id =
                                        String(
                                            validation
                                                .targetClusterId
                                        );

                                    output.former_cluster_id =
                                        String(
                                            formerClusterId
                                        );

                                    output.cluster_creation_method =
                                        "dbscan_manual_merge";
                                }} else {{
                                    output.final_decision =
                                        "pending-manual-decision";

                                    output.final_cluster_id =
                                        "";

                                    output.former_cluster_id =
                                        String(
                                            formerClusterId
                                        );

                                    output.cluster_creation_method =
                                        "";
                                }}
                            }} else if (
                                decision
                                === "create-separate-final-clusters"
                            ) {{
                                const assignment =
                                    getAssignment(
                                        formerClusterId,
                                        output.crop_id
                                    );

                                output.former_cluster_id =
                                    String(
                                        formerClusterId
                                    );

                                if (
                                    assignment === null
                                ) {{
                                    output.final_decision =
                                        "pending-manual-decision";

                                    output.final_cluster_id =
                                        "";

                                    output.cluster_creation_method =
                                        "";
                                }} else if (
                                    assignment === 0
                                ) {{
                                    output.final_decision =
                                        "excluded-from-corpus";

                                    output.final_cluster_id =
                                        "";

                                    output.cluster_creation_method =
                                        "";
                                }} else {{
                                    output.final_decision =
                                        characterCategory;

                                    output.final_cluster_id =
                                        provisionalFinalClusterId(
                                            formerClusterId,
                                            assignment
                                        );

                                    output.cluster_creation_method =
                                        "dbscan_manual_intervention";
                                }}
                            }} else {{
                                output.final_decision =
                                    "pending-manual-decision";

                                output.final_cluster_id =
                                    "";

                                output.former_cluster_id =
                                    String(
                                        formerClusterId
                                    );

                                output.cluster_creation_method =
                                    "";
                            }}
                        }}

                        return output;
                    }}
                );


            const addedColumns = [
                "former_cluster_id",
                "final_decision",
                "final_cluster_id",
                "cluster_creation_method"
            ];

            const exportColumns = [
                ...originalColumns.filter(
                    column =>
                        !addedColumns.includes(
                            column
                        )
                ),
                ...addedColumns
            ];


            const rows = [
                exportColumns
            ];

            finalRows.forEach(
                row => {{
                    rows.push(
                        exportColumns.map(
                            column =>
                                row[column] ?? ""
                        )
                    );
                }}
            );


            const csvText =
                rows
                .map(
                    row =>
                        row
                        .map(
                            csvEscape
                        )
                        .join(",")
                )
                .join("\\n");


            const blob =
                new Blob(
                    [csvText],
                    {{
                        type:
                            "text/csv;charset=utf-8"
                    }}
                );

            const url =
                URL.createObjectURL(
                    blob
                );

            const link =
                document.createElement(
                    "a"
                );

            link.href =
                url;

            link.download =
                "cluster_classifications_final.csv";

            document.body.appendChild(
                link
            );

            link.click();

            link.remove();

            URL.revokeObjectURL(
                url
            );
        }}


        // -------------------------------------------------
        // Decision-button event listeners
        // -------------------------------------------------

        document.getElementById(
            "keep-button"
        ).addEventListener(
            "click",
            () => {{
                const cluster =
                    clusters[
                        currentClusterPosition
                    ];

                setDecision(
                    cluster.former_cluster_id,
                    "keep-as-character-cluster"
                );
            }}
        );


        document.getElementById(
            "divide-button"
        ).addEventListener(
            "click",
            () => {{
                const cluster =
                    clusters[
                        currentClusterPosition
                    ];

                setDecision(
                    cluster.former_cluster_id,
                    "create-separate-final-clusters"
                );
            }}
        );


        document.getElementById(
            "merge-button"
        ).addEventListener(
            "click",
            () => {{
                const cluster =
                    clusters[
                        currentClusterPosition
                    ];

                setDecision(
                    cluster.former_cluster_id,
                    "add-to-existing-cluster"
                );
            }}
        );


        document.getElementById(
            "reject-button"
        ).addEventListener(
            "click",
            () => {{
                const cluster =
                    clusters[
                        currentClusterPosition
                    ];

                setDecision(
                    cluster.former_cluster_id,
                    "not-a-character-cluster"
                );
            }}
        );


        // -------------------------------------------------
        // Merge-target event listeners
        // -------------------------------------------------

        document.getElementById(
            "save-merge-target-button"
        ).addEventListener(
            "click",
            () => {{
                const cluster =
                    clusters[
                        currentClusterPosition
                    ];

                const input =
                    document.getElementById(
                        "merge-target-input"
                    );

                const validation =
                    validateMergeTarget(
                        cluster.former_cluster_id,
                        input.value
                    );

                const message =
                    document.getElementById(
                        "merge-validation-message"
                    );

                message.textContent =
                    validation.message;

                message.classList.toggle(
                    "warning",
                    !validation.valid
                );

                message.classList.toggle(
                    "success",
                    validation.valid
                );

                if (
                    !validation.valid
                ) {{
                    return;
                }}

                setMergeTarget(
                    cluster.former_cluster_id,
                    validation.targetClusterId
                );

                renderCurrentCluster();
            }}
        );


        document.getElementById(
            "merge-target-input"
        ).addEventListener(
            "keydown",
            event => {{
                if (
                    event.key === "Enter"
                ) {{
                    event.preventDefault();

                    document.getElementById(
                        "save-merge-target-button"
                    ).click();
                }}
            }}
        );


        // -------------------------------------------------
        // Manual division event listeners
        // -------------------------------------------------

        document.getElementById(
            "exclude-button"
        ).addEventListener(
            "click",
            () => {{
                activeFinalClusterNumber = 0;

                renderCurrentCluster();
            }}
        );


        document.getElementById(
            "add-cluster-button"
        ).addEventListener(
            "click",
            () => {{
                const cluster =
                    clusters[
                        currentClusterPosition
                    ];

                const currentCount =
                    getFinalClusterCount(
                        cluster.former_cluster_id
                    );

                const newCount =
                    currentCount + 1;

                setFinalClusterCount(
                    cluster.former_cluster_id,
                    newCount
                );

                activeFinalClusterNumber =
                    newCount;

                renderCurrentCluster();
            }}
        );


        // -------------------------------------------------
        // Navigation event listeners
        // -------------------------------------------------

        document.getElementById(
            "previous-button"
        ).addEventListener(
            "click",
            () => {{
                if (
                    currentClusterPosition > 0
                ) {{
                    currentClusterPosition--;

                    activeFinalClusterNumber = 1;

                    renderCurrentCluster();
                }}
            }}
        );


        document.getElementById(
            "next-button"
        ).addEventListener(
            "click",
            () => {{
                if (
                    currentClusterPosition
                    < clusters.length - 1
                ) {{
                    currentClusterPosition++;

                    activeFinalClusterNumber = 1;

                    renderCurrentCluster();
                }}
            }}
        );


        // -------------------------------------------------
        // Export event listener
        // -------------------------------------------------

        document.getElementById(
            "export-button"
        ).addEventListener(
            "click",
            exportFinalReport
        );


        // -------------------------------------------------
        // Initial rendering
        // -------------------------------------------------

        renderCurrentCluster();
    </script>
</body>
</html>
"""


# -----------------------------------------------------
# Save HTML
# -----------------------------------------------------

FINAL_CLUSTER_REPORT_HTML_PATH.write_text(
    final_cluster_report_html,
    encoding="utf-8",
)

report_size_mb = (
    FINAL_CLUSTER_REPORT_HTML_PATH.stat().st_size
    / 1024
    / 1024
)


# -----------------------------------------------------
# Print report summary
# -----------------------------------------------------

confirmed_character_clusters = int(
    (
        cluster_classifications[
            "cluster_category"
        ]
        == CHARACTER_CATEGORY
    ).sum()
)

not_character_clusters = int(
    (
        cluster_classifications[
            "cluster_category"
        ]
        == NOT_CHARACTER_CATEGORY
    ).sum()
)

manual_rework_clusters = int(
    (
        cluster_classifications[
            "cluster_category"
        ]
        == MANUAL_REWORK_CATEGORY
    ).sum()
)

unspecified_clusters = int(
    (
        cluster_classifications[
            "cluster_category"
        ]
        == ""
    ).sum()
)

valid_merge_target_count = int(
    len(
        [
            cluster_id
            for (
                cluster_id,
                category,
            ) in cluster_category_lookup.items()
            if category
            == CHARACTER_CATEGORY
        ]
    )
)

print("=" * 70)
print("FINAL CLUSTER-CURATION REPORT CREATED")
print("=" * 70)

print(
    "Confirmed character clusters:",
    f"{confirmed_character_clusters:,}",
)

print(
    "Not-a-character clusters:",
    f"{not_character_clusters:,}",
)

print(
    "Clusters requiring manual decisions:",
    f"{manual_rework_clusters:,}",
)

print(
    "Unspecified clusters treated as noise:",
    f"{unspecified_clusters:,}",
)

print(
    "Valid existing merge targets:",
    f"{valid_merge_target_count:,}",
)

print(
    "Crops requiring manual inspection:",
    f"{len(manual_images):,}",
)

print(
    "Rows in crop-level final-report base:",
    f"{len(base_report):,}",
)

print(
    f"HTML size: {report_size_mb:.2f} MB"
)

print("Saved to:")
print(FINAL_CLUSTER_REPORT_HTML_PATH)

print()
print(
    "The HTML export will download:"
)

print(
    "cluster_classifications_final.csv"
)

print("=" * 70)


# -----------------------------------------------------
# Display manually reviewed clusters
# -----------------------------------------------------

display(
    cluster_classifications.loc[
        cluster_classifications[
            "cluster_id"
        ].isin(
            manual_cluster_ids
        )
    ]
    .sort_values(
        "cluster_id"
    )
    .reset_index(
        drop=True
    )
)

##6) Reviewing duplicate crops of the same character per page

This cell implements one last check before the final export. Characters only appear once per page - this cell makes sure no crops inside the same cluster refer to the same page. If there are duplicates of the same print, they are provided side by side, with the possibility to decide which crop should be kept and which one is to be disregarded.

In [ ]:
# =====================================================
# Review duplicate crops of the same character per page
# =====================================================

PIPELINE_ROOT = Path(PIPELINE_ROOT)
CLUSTERING_VARIANT_DIR = Path(CLUSTERING_VARIANT_DIR)

FINAL_CURATION_CSV_PATH = (
    CLUSTERING_VARIANT_DIR
    / "cluster_classifications_final.csv"
)


# -----------------------------------------------------
# Helper functions
# -----------------------------------------------------

def normalize_boolean_value(value):
    """
    Convert common CSV representations into True, False,
    or pd.NA.
    """
    if pd.isna(value):
        return pd.NA

    normalized = str(value).strip().lower()

    if normalized in {
        "true",
        "1",
        "yes",
        "y",
        "keep",
    }:
        return True

    if normalized in {
        "false",
        "0",
        "no",
        "n",
        "remove",
    }:
        return False

    if normalized == "":
        return pd.NA

    return pd.NA


def resolve_crop_path(path_value):
    """
    Resolve an absolute crop path or a path relative to PIPELINE_ROOT.
    """
    if pd.isna(path_value):
        return None

    path_text = str(path_value).strip()

    if not path_text:
        return None

    crop_path = Path(path_text)

    if crop_path.is_absolute():
        return crop_path

    return PIPELINE_ROOT / crop_path


def save_final_curation_csv(dataframe):
    """
    Save the current review state.
    """
    dataframe.to_csv(
        FINAL_CURATION_CSV_PATH,
        index=False,
    )


def display_duplicate_group(
    group_dataframe,
    group_number,
    total_groups,
):
    """
    Display every candidate crop in one duplicate group.
    """
    final_cluster_id = (
        group_dataframe["final_cluster_id"].iloc[0]
    )

    page_id = (
        group_dataframe["page_id"].iloc[0]
    )

    number_of_crops = len(group_dataframe)

    print("=" * 80)
    print(
        f"Duplicate group {group_number:,} "
        f"of {total_groups:,}"
    )
    print("=" * 80)
    print(f"Final character cluster: {final_cluster_id}")
    print(f"Page:                    {page_id}")
    print(f"Candidate crops:         {number_of_crops}")
    print("=" * 80)

    figure_width = max(
        5 * number_of_crops,
        8,
    )

    figure, axes = plt.subplots(
        1,
        number_of_crops,
        figsize=(figure_width, 6),
        squeeze=False,
    )

    axes = axes.flatten()

    for option_number, (_, row) in enumerate(
        group_dataframe.iterrows(),
        start=1,
    ):
        axis = axes[option_number - 1]

        crop_id = row["crop_id"]

        crop_path = resolve_crop_path(
            row["local_crop_path"]
        )

        image_loaded = False

        if (
            crop_path is not None
            and crop_path.exists()
        ):
            try:
                with Image.open(crop_path) as image:
                    display_image = image.convert("RGB")
                    axis.imshow(display_image)

                image_loaded = True

            except (
                OSError,
                UnidentifiedImageError,
            ) as error:
                axis.text(
                    0.5,
                    0.5,
                    "Image could not be opened",
                    horizontalalignment="center",
                    verticalalignment="center",
                    wrap=True,
                )

                print(
                    f"WARNING: Could not open {crop_id}:\n"
                    f"{crop_path}\n"
                    f"{error}"
                )

        if not image_loaded:
            if (
                crop_path is None
                or not crop_path.exists()
            ):
                axis.text(
                    0.5,
                    0.5,
                    "Image file not found",
                    horizontalalignment="center",
                    verticalalignment="center",
                    wrap=True,
                )

                print(
                    f"WARNING: Missing crop image for "
                    f"{crop_id}:\n"
                    f"{crop_path}"
                )

        axis.set_title(
            f"Option {option_number}\n{crop_id}",
            fontsize=11,
        )

        axis.axis("off")

    plt.tight_layout()
    plt.show()

    print()
    print("Candidate crops:")

    for option_number, (_, row) in enumerate(
        group_dataframe.iterrows(),
        start=1,
    ):
        crop_path = resolve_crop_path(
            row["local_crop_path"]
        )

        print(
            f"  {option_number}: {row['crop_id']}"
        )

        print(
            f"     Path: {crop_path}"
        )


# -----------------------------------------------------
# Load final curation CSV
# -----------------------------------------------------

if not FINAL_CURATION_CSV_PATH.exists():
    raise FileNotFoundError(
        "The final curation CSV was not found:\n"
        f"{FINAL_CURATION_CSV_PATH}"
    )

final_curation = pd.read_csv(
    FINAL_CURATION_CSV_PATH,
    dtype=str,
)

final_curation.columns = (
    final_curation.columns
    .astype(str)
    .str.strip()
)


# -----------------------------------------------------
# Validate required columns
# -----------------------------------------------------

required_columns = {
    "crop_id",
    "page_id",
    "final_cluster_id",
    "final_decision",
    "local_crop_path",
}

missing_columns = (
    required_columns
    - set(final_curation.columns)
)

if missing_columns:
    raise ValueError(
        "The final curation CSV is missing required columns:\n"
        f"{sorted(missing_columns)}"
    )


# -----------------------------------------------------
# Normalize relevant text columns
# -----------------------------------------------------

columns_to_normalize = [
    "crop_id",
    "page_id",
    "final_cluster_id",
    "final_decision",
    "local_crop_path",
    "former_cluster_id",
    "cluster_creation_method",
]

for column in columns_to_normalize:
    if column in final_curation.columns:
        final_curation[column] = (
            final_curation[column]
            .fillna("")
            .astype(str)
            .str.strip()
        )

final_curation["final_decision"] = (
    final_curation["final_decision"]
    .str.lower()
)


# -----------------------------------------------------
# Add or normalize keep_crop
# -----------------------------------------------------

if "keep_crop" not in final_curation.columns:
    # Retain all crops initially.
    #
    # Unselected alternatives will later be changed
    # from True to False.
    final_curation["keep_crop"] = True

else:
    normalized_keep_crop = (
        final_curation["keep_crop"]
        .map(normalize_boolean_value)
    )

    # Empty or unknown values are treated as True.
    final_curation["keep_crop"] = (
        normalized_keep_crop
        .fillna(True)
        .astype(bool)
    )


# -----------------------------------------------------
# Validate crop IDs
# -----------------------------------------------------

missing_crop_ids = final_curation.loc[
    final_curation["crop_id"] == ""
]

if not missing_crop_ids.empty:
    raise ValueError(
        "The final curation CSV contains rows without crop_id."
    )

duplicated_crop_ids = final_curation.loc[
    final_curation["crop_id"].duplicated(
        keep=False
    ),
    "crop_id",
].drop_duplicates()

if not duplicated_crop_ids.empty:
    raise ValueError(
        "The final curation CSV contains duplicated crop_id "
        "values. Each crop_id must occur exactly once.\n\n"
        f"Examples:\n"
        f"{duplicated_crop_ids.head(20).tolist()}"
    )


# -----------------------------------------------------
# Select accepted character crops
# -----------------------------------------------------

accepted_mask = (
    (
        final_curation["final_decision"]
        == "character-cluster"
    )
    & (
        final_curation["final_cluster_id"]
        != ""
    )
)

accepted_crops = final_curation.loc[
    accepted_mask
].copy()

if accepted_crops.empty:
    raise ValueError(
        "No accepted character crops were found.\n\n"
        "Expected rows with:\n"
        'final_decision = "character-cluster"\n'
        "and a non-empty final_cluster_id."
    )

missing_page_rows = accepted_crops.loc[
    accepted_crops["page_id"] == ""
]

if not missing_page_rows.empty:
    raise ValueError(
        "Some accepted character crops have no page_id.\n\n"
        "Example crop IDs:\n"
        f"{missing_page_rows['crop_id'].head(20).tolist()}"
    )


# -----------------------------------------------------
# Detect duplicate character/page groups
# -----------------------------------------------------

duplicate_group_columns = [
    "final_cluster_id",
    "page_id",
]

duplicate_group_sizes = (
    accepted_crops
    .groupby(
        duplicate_group_columns,
        dropna=False,
    )
    .size()
    .rename("candidate_count")
    .reset_index()
)

duplicate_group_keys = (
    duplicate_group_sizes.loc[
        duplicate_group_sizes["candidate_count"] > 1,
        duplicate_group_columns,
    ]
    .copy()
)

# Preserve the original row indices from final_curation.
accepted_crops["_source_index"] = (
    accepted_crops.index
)

duplicate_rows = (
    accepted_crops
    .merge(
        duplicate_group_keys,
        on=duplicate_group_columns,
        how="inner",
        validate="many_to_one",
    )
    .sort_values(
        [
            "final_cluster_id",
            "page_id",
            "crop_id",
        ]
    )
    .reset_index(drop=True)
)

duplicate_groups = list(
    duplicate_rows.groupby(
        duplicate_group_columns,
        sort=True,
        dropna=False,
    )
)


# -----------------------------------------------------
# Classify review status
# -----------------------------------------------------

completed_groups = []
unresolved_groups = []
inconsistent_groups = []

for group_key, group in duplicate_groups:
    source_indices = (
        group["_source_index"]
        .astype(int)
        .tolist()
    )

    keep_values = (
        final_curation.loc[
            source_indices,
            "keep_crop",
        ]
        .astype(bool)
    )

    kept_count = int(keep_values.sum())
    group_size = len(group)

    if kept_count == 1:
        completed_groups.append(
            (
                group_key,
                group,
            )
        )

    elif kept_count == group_size:
        # All candidates still have their initial True value.
        unresolved_groups.append(
            (
                group_key,
                group,
            )
        )

    else:
        # Examples:
        # - every candidate is False
        # - more than one but not all candidates are True
        #
        # These groups are shown again so the values can be repaired.
        inconsistent_groups.append(
            (
                group_key,
                group,
            )
        )


# -----------------------------------------------------
# Save keep_crop before starting review
# -----------------------------------------------------

save_final_curation_csv(final_curation)


# -----------------------------------------------------
# Initial summary
# -----------------------------------------------------

print("=" * 80)
print("DUPLICATE CROP REVIEW")
print("=" * 80)

print(
    "Accepted character crops:",
    f"{len(accepted_crops):,}",
)

print(
    "Final character clusters:",
    f"{accepted_crops['final_cluster_id'].nunique():,}",
)

print(
    "Duplicate character/page groups:",
    f"{len(duplicate_groups):,}",
)

print(
    "Already completed groups:",
    f"{len(completed_groups):,}",
)

print(
    "Unreviewed groups:",
    f"{len(unresolved_groups):,}",
)

print(
    "Groups with inconsistent prior values:",
    f"{len(inconsistent_groups):,}",
)

print()
print("Working CSV:")
print(FINAL_CURATION_CSV_PATH)

print("=" * 80)


# Review inconsistent groups first, followed by
# completely unreviewed groups.
groups_to_review = (
    inconsistent_groups
    + unresolved_groups
)


# -----------------------------------------------------
# Interactive visual review
# -----------------------------------------------------

review_interrupted = False

if not duplicate_groups:
    print(
        "No accepted character appears more than once "
        "on the same page."
    )

elif not groups_to_review:
    print(
        "All same-page duplicate groups have already "
        "been reviewed."
    )

else:
    total_groups_to_review = len(groups_to_review)

    for review_number, (
        group_key,
        group,
    ) in enumerate(
        groups_to_review,
        start=1,
    ):
        print()
        print()
        print("#" * 80)

        print(
            f"STARTING REVIEW GROUP "
            f"{review_number:,} OF "
            f"{total_groups_to_review:,}"
        )

        print("#" * 80)
        print()

        display_duplicate_group(
            group_dataframe=group,
            group_number=review_number,
            total_groups=total_groups_to_review,
        )

        crop_ids = (
            group["crop_id"]
            .astype(str)
            .tolist()
        )

        option_lookup = {
            str(option_number): crop_id
            for option_number, crop_id
            in enumerate(
                crop_ids,
                start=1,
            )
        }

        valid_crop_ids = set(crop_ids)

        print()
        print(
            "Enter the option number or the full crop_id "
            "that should be kept."
        )

        print(
            "Enter Q to save and stop the review."
        )

        while True:
            user_choice = input(
                "\nCrop to keep: "
            ).strip()

            if user_choice.lower() in {
                "q",
                "quit",
                "stop",
                "exit",
            }:
                save_final_curation_csv(
                    final_curation
                )

                review_interrupted = True

                print()
                print(
                    "Review stopped. All completed decisions "
                    "have been saved."
                )

                break

            if user_choice in option_lookup:
                selected_crop_id = (
                    option_lookup[user_choice]
                )

            elif user_choice in valid_crop_ids:
                selected_crop_id = user_choice

            else:
                print(
                    "Invalid selection. Enter one of the "
                    "displayed option numbers or crop IDs."
                )

                continue

            source_indices = (
                group["_source_index"]
                .astype(int)
                .tolist()
            )

            # Exclude every crop in this group.
            final_curation.loc[
                source_indices,
                "keep_crop",
            ] = False

            selected_source_indices = (
                group.loc[
                    group["crop_id"]
                    == selected_crop_id,
                    "_source_index",
                ]
                .astype(int)
                .tolist()
            )

            if len(selected_source_indices) != 1:
                raise ValueError(
                    "The selected crop could not be mapped "
                    "unambiguously to one CSV row:\n"
                    f"{selected_crop_id}"
                )

            # Retain exactly the selected crop.
            final_curation.loc[
                selected_source_indices,
                "keep_crop",
            ] = True

            # Save immediately after every decision.
            save_final_curation_csv(
                final_curation
            )

            removed_crop_ids = [
                crop_id
                for crop_id in crop_ids
                if crop_id != selected_crop_id
            ]

            print()
            print(
                f"Kept: {selected_crop_id}"
            )

            print(
                "Excluded:",
                ", ".join(removed_crop_ids),
            )

            print(
                "Decision saved."
            )

            break

        if review_interrupted:
            break


# -----------------------------------------------------
# Final validation
# -----------------------------------------------------

validation_records = []

for group_key, group in duplicate_groups:
    source_indices = (
        group["_source_index"]
        .astype(int)
        .tolist()
    )

    kept_count = int(
        final_curation.loc[
            source_indices,
            "keep_crop",
        ]
        .astype(bool)
        .sum()
    )

    validation_records.append(
        {
            "final_cluster_id": group_key[0],
            "page_id": group_key[1],
            "candidate_count": len(group),
            "kept_count": kept_count,
        }
    )

duplicate_validation = pd.DataFrame(
    validation_records
)

if duplicate_validation.empty:
    incomplete_groups = duplicate_validation.copy()

else:
    incomplete_groups = (
        duplicate_validation.loc[
            duplicate_validation["kept_count"] != 1
        ]
        .copy()
    )


# -----------------------------------------------------
# Final summary
# -----------------------------------------------------

accepted_crops_updated = final_curation.loc[
    accepted_mask
].copy()

kept_accepted_crops = int(
    accepted_crops_updated["keep_crop"]
    .astype(bool)
    .sum()
)

excluded_accepted_crops = int(
    (
        ~accepted_crops_updated["keep_crop"]
        .astype(bool)
    )
    .sum()
)

print()
print()
print("=" * 80)
print("CURRENT REVIEW STATUS")
print("=" * 80)

print(
    "Accepted character crops before duplicate removal:",
    f"{len(accepted_crops_updated):,}",
)

print(
    "Accepted character crops currently retained:",
    f"{kept_accepted_crops:,}",
)

print(
    "Duplicate alternatives currently excluded:",
    f"{excluded_accepted_crops:,}",
)

print(
    "Duplicate groups still requiring review:",
    f"{len(incomplete_groups):,}",
)

print()
print("Updated final curation CSV:")
print(FINAL_CURATION_CSV_PATH)

if incomplete_groups.empty:
    print()
    print(
        "Review complete: every same-page duplicate group "
        "has exactly one retained crop."
    )

    print()
    print(
        "The corpus export can now select rows where:"
    )

    print(
        'final_decision == "character-cluster"'
    )

    print(
        "and keep_crop == True"
    )

else:
    print()
    print(
        "The review is not yet complete."
    )

    print(
        "Run this cell again to continue with the "
        "remaining groups."
    )

print("=" * 80)

##7) Export of the finalized Corpus

After reviewing the Clusters created via the embedding and the DBSCAN implementation, the finalized Corpus can be exported by using the created export-csv file.

 The cell creates the following directory-structure within the pipeline-root:

 Corpus/

 ├── Clusters/

 │   ├── cluster_0001/

 │   ├── cluster_0002/

 │   └── ...

 ├── Pages_JPG/

 └── Metadata/

     ├── character_corpus_metadata.csv

     ├── cluster_summary.csv

     └── export_report.txt


In [ ]:
# -----------------------------------------------------
# Path Configuration
# -----------------------------------------------------

FINAL_CURATION_CSV_PATH = (
    PIPELINE_ROOT
    / "Embeddings/datacomp_vitl14_dbscan/clustering/dbscan_eps008_min3_cosine/cluster_classifications_final.csv"
)

CROP_METADATA_CSV_PATH = (
    PIPELINE_ROOT
    / "Data/Crops_Bombe/metadata/Crops_Bombe_metadata.csv"
)

PAGE_METADATA_CSV_PATH = (
    PIPELINE_ROOT
    / "Data/Data_acquisition_BOMBE/metadata/pages-jpg_metadata.csv"
)

CORPUS_ROOT = (
    PIPELINE_ROOT
    / "Corpus"
)

# When True, the generated directories are rebuilt
RESET_EXISTING_EXPORT = True

# Produces cluster_0001, cluster_0002, etc.
CLUSTER_ID_WIDTH = 4


# -----------------------------------------------------
# Output paths
# -----------------------------------------------------

CORPUS_CLUSTERS_DIR = (
    CORPUS_ROOT
    / "Clusters"
)

CORPUS_PAGES_DIR = (
    CORPUS_ROOT
    / "Pages_JPG"
)

CORPUS_METADATA_DIR = (
    CORPUS_ROOT
    / "Metadata"
)

CORPUS_METADATA_PATH = (
    CORPUS_METADATA_DIR
    / "character_corpus_metadata.csv"
)

CLUSTER_SUMMARY_PATH = (
    CORPUS_METADATA_DIR
    / "cluster_summary.csv"
)

EXPORT_REPORT_PATH = (
    CORPUS_METADATA_DIR
    / "export_report.txt"
)


# -----------------------------------------------------
# Helper functions
# -----------------------------------------------------

def normalize_boolean(value):
    """
    Convert common CSV boolean values into True or False.
    """
    if pd.isna(value):
        return False

    normalized = str(value).strip().lower()

    if normalized in {
        "true",
        "1",
        "yes",
        "y",
        "keep",
    }:
        return True

    if normalized in {
        "false",
        "0",
        "no",
        "n",
        "remove",
        "",
    }:
        return False

    raise ValueError(
        "Unrecognized boolean value in keep_crop: "
        f"{value!r}"
    )


def normalize_text_columns(
    dataframe,
    columns,
):
    """
    Normalize selected text columns.
    """
    dataframe = dataframe.copy()

    for column in columns:
        if column in dataframe.columns:
            dataframe[column] = (
                dataframe[column]
                .fillna("")
                .astype(str)
                .str.strip()
            )

    return dataframe


def resolve_source_path(
    path_value,
    base_directory,
):
    """
    Resolve an absolute path or a path relative to
    the pipeline root.
    """
    if pd.isna(path_value):
        return None

    path_text = str(path_value).strip()

    if not path_text:
        return None

    source_path = Path(path_text)

    if source_path.is_absolute():
        return source_path

    return (
        Path(base_directory)
        / source_path
    )


def natural_sort_key(value):
    """
    Sort numeric and mixed curated cluster IDs naturally.
    """
    text = str(value).strip()

    parts = re.split(
        r"(\d+)",
        text,
    )

    key = []

    for part in parts:
        if part.isdigit():
            key.append(
                (
                    0,
                    int(part),
                )
            )
        else:
            key.append(
                (
                    1,
                    part.lower(),
                )
            )

    return key


def safe_filename(value):
    """
    Return only the filename component of a path.
    """
    return Path(
        str(value)
    ).name


def portable_path(path):
    """
    Return a forward-slash relative path.
    """
    return Path(path).as_posix()


# -----------------------------------------------------
# Validate input paths
# -----------------------------------------------------

input_paths = {
    "Final curation CSV":
        FINAL_CURATION_CSV_PATH,

    "Crop metadata CSV":
        CROP_METADATA_CSV_PATH,

    "Page metadata CSV":
        PAGE_METADATA_CSV_PATH,
}

missing_input_paths = [
    (
        label,
        path,
    )
    for (
        label,
        path,
    ) in input_paths.items()
    if not Path(path).exists()
]

if missing_input_paths:
    formatted_paths = "\n".join(
        f"- {label}: {path}"
        for label, path
        in missing_input_paths
    )

    raise FileNotFoundError(
        "One or more required input files were not found:\n"
        f"{formatted_paths}"
    )


# -----------------------------------------------------
# Load source metadata
# -----------------------------------------------------

final_curation = pd.read_csv(
    FINAL_CURATION_CSV_PATH,
    dtype=str,
)

crop_metadata = pd.read_csv(
    CROP_METADATA_CSV_PATH,
    dtype=str,
)

page_metadata = pd.read_csv(
    PAGE_METADATA_CSV_PATH,
    dtype=str,
)

for dataframe in [
    final_curation,
    crop_metadata,
    page_metadata,
]:
    dataframe.columns = (
        dataframe.columns
        .astype(str)
        .str.strip()
    )


# -----------------------------------------------------
# Validate required source columns
# -----------------------------------------------------

required_curation_columns = {
    "crop_id",
    "final_cluster_id",
    "final_decision",
    "keep_crop",
}

required_crop_columns = {
    "crop_id",
    "page_id",
    "local_crop_path",
    "anno_id",
    "iiif_crop_url",
    "iiif_page_url",
}

required_page_columns = {
    "page_id",
    "page_number",
    "local_jpg_path",
    "anno_id",
    "date",
    "periodical_title",
    "publishing_place",
    "publishing_place_gnd",
    "language",
    "mediatype",
    "attribution",
    "provider",
    "rights",
}

missing_curation_columns = (
    required_curation_columns
    - set(final_curation.columns)
)

missing_crop_columns = (
    required_crop_columns
    - set(crop_metadata.columns)
)

missing_page_columns = (
    required_page_columns
    - set(page_metadata.columns)
)

validation_errors = []

if missing_curation_columns:
    validation_errors.append(
        "Final curation CSV is missing:\n"
        f"{sorted(missing_curation_columns)}"
    )

if missing_crop_columns:
    validation_errors.append(
        "Crop metadata is missing:\n"
        f"{sorted(missing_crop_columns)}"
    )

if missing_page_columns:
    validation_errors.append(
        "Page metadata is missing:\n"
        f"{sorted(missing_page_columns)}"
    )

if validation_errors:
    raise ValueError(
        "\n\n".join(
            validation_errors
        )
    )


# -----------------------------------------------------
# Normalize relevant fields
# -----------------------------------------------------

final_curation = normalize_text_columns(
    final_curation,
    [
        "crop_id",
        "final_cluster_id",
        "final_decision",
        "keep_crop",
    ],
)

crop_metadata = normalize_text_columns(
    crop_metadata,
    [
        "crop_id",
        "page_id",
        "local_crop_path",
        "anno_id",
        "iiif_crop_url",
        "iiif_page_url",
    ],
)

page_metadata = normalize_text_columns(
    page_metadata,
    [
        "page_id",
        "page_number",
        "local_jpg_path",
        "anno_id",
        "date",
        "periodical_title",
        "publishing_place",
        "publishing_place_gnd",
        "language",
        "mediatype",
        "attribution",
        "provider",
        "rights",
    ],
)

final_curation["final_decision"] = (
    final_curation["final_decision"]
    .str.lower()
)

final_curation["keep_crop"] = (
    final_curation["keep_crop"]
    .map(normalize_boolean)
)


# -----------------------------------------------------
# Validate unique identifiers
# -----------------------------------------------------

for dataframe_name, dataframe, id_column in [
    (
        "final curation CSV",
        final_curation,
        "crop_id",
    ),
    (
        "crop metadata",
        crop_metadata,
        "crop_id",
    ),
    (
        "page metadata",
        page_metadata,
        "page_id",
    ),
]:
    duplicated_ids = (
        dataframe.loc[
            dataframe[id_column].duplicated(
                keep=False
            ),
            id_column,
        ]
        .drop_duplicates()
        .tolist()
    )

    if duplicated_ids:
        raise ValueError(
            f"The {dataframe_name} contains duplicated "
            f"{id_column} values:\n"
            f"{duplicated_ids[:20]}"
        )


# -----------------------------------------------------
# Select retained character crops
# -----------------------------------------------------

accepted_before_keep_mask = (
    (
        final_curation["final_decision"]
        == "character-cluster"
    )
    & (
        final_curation["final_cluster_id"]
        != ""
    )
)

export_selection_mask = (
    accepted_before_keep_mask
    & final_curation["keep_crop"]
)

accepted_before_keep = (
    final_curation.loc[
        accepted_before_keep_mask
    ]
    .copy()
)

selected_curation = (
    final_curation.loc[
        export_selection_mask,
        [
            "crop_id",
            "final_cluster_id",
        ],
    ]
    .copy()
)

if selected_curation.empty:
    raise ValueError(
        "No crops satisfy both export conditions:\n"
        'final_decision == "character-cluster"\n'
        "keep_crop == True"
    )


# -----------------------------------------------------
# Join retained crops to crop metadata
# -----------------------------------------------------

export_metadata = selected_curation.merge(
    crop_metadata,
    on="crop_id",
    how="left",
    validate="one_to_one",
    indicator="_crop_join",
)

missing_crop_metadata = export_metadata.loc[
    export_metadata["_crop_join"] != "both"
].copy()

if not missing_crop_metadata.empty:
    raise ValueError(
        "Some retained crops are missing from crop metadata:\n"
        f"{missing_crop_metadata['crop_id'].head(20).tolist()}"
    )

export_metadata = export_metadata.drop(
    columns=[
        "_crop_join",
    ]
)


# -----------------------------------------------------
# Join retained crops to page metadata
# -----------------------------------------------------

export_metadata = export_metadata.rename(
    columns={
        "anno_id":
            "crop_issue_id",
    }
)

page_columns_for_join = [
    "page_id",
    "page_number",
    "local_jpg_path",
    "anno_id",
    "date",
    "periodical_title",
    "publishing_place",
    "publishing_place_gnd",
    "language",
    "mediatype",
    "attribution",
    "provider",
    "rights",
]

export_metadata = export_metadata.merge(
    page_metadata[
        page_columns_for_join
    ],
    on="page_id",
    how="left",
    validate="many_to_one",
    indicator="_page_join",
)

missing_page_metadata = export_metadata.loc[
    export_metadata["_page_join"] != "both"
].copy()

if not missing_page_metadata.empty:
    raise ValueError(
        "Some retained crops refer to pages missing from "
        "page metadata:\n"
        f"{missing_page_metadata['page_id'].drop_duplicates().head(20).tolist()}"
    )

export_metadata = export_metadata.drop(
    columns=[
        "_page_join",
    ]
)


# -----------------------------------------------------
# Validate issue identifiers
# -----------------------------------------------------

issue_mismatches = export_metadata.loc[
    (
        export_metadata["crop_issue_id"]
        != export_metadata["anno_id"]
    )
    & (
        export_metadata["crop_issue_id"]
        != ""
    )
    & (
        export_metadata["anno_id"]
        != ""
    )
].copy()

if not issue_mismatches.empty:
    mismatch_examples = (
        issue_mismatches[
            [
                "crop_id",
                "page_id",
                "crop_issue_id",
                "anno_id",
            ]
        ]
        .head(20)
        .to_dict(orient="records")
    )

    raise ValueError(
        "Issue identifiers disagree between crop and "
        "page metadata.\n\n"
        f"Examples:\n{mismatch_examples}"
    )


# -----------------------------------------------------
# Validate one retained crop per character/page
# -----------------------------------------------------

same_page_counts = (
    export_metadata
    .groupby(
        [
            "final_cluster_id",
            "page_id",
        ],
        dropna=False,
    )
    .size()
)

same_page_conflicts = same_page_counts.loc[
    same_page_counts > 1
]

if not same_page_conflicts.empty:
    raise ValueError(
        "More than one retained crop still exists for the "
        "same character and page:\n"
        f"{same_page_conflicts.head(20)}"
    )


# -----------------------------------------------------
# Create ordered consecutive corpus cluster IDs
# -----------------------------------------------------

curated_cluster_ids = (
    export_metadata["final_cluster_id"]
    .drop_duplicates()
    .tolist()
)

curated_cluster_ids = sorted(
    curated_cluster_ids,
    key=natural_sort_key,
)

cluster_id_mapping = {
    curated_cluster_id: (
        f"cluster_"
        f"{number:0{CLUSTER_ID_WIDTH}d}"
    )
    for number, curated_cluster_id
    in enumerate(
        curated_cluster_ids,
        start=1,
    )
}

export_metadata["cluster_id"] = (
    export_metadata["final_cluster_id"]
    .map(cluster_id_mapping)
)

if export_metadata["cluster_id"].isna().any():
    raise ValueError(
        "At least one curated cluster could not be mapped "
        "to a new corpus cluster ID."
    )


# -----------------------------------------------------
# Build corpus-relative paths
# -----------------------------------------------------

export_metadata["crop_filename"] = (
    export_metadata["local_crop_path"]
    .map(safe_filename)
)

export_metadata["page_filename"] = (
    export_metadata["local_jpg_path"]
    .map(safe_filename)
)

export_metadata["crop_path"] = (
    export_metadata.apply(
        lambda row:
            portable_path(
                Path("Clusters")
                / row["cluster_id"]
                / row["crop_filename"]
            ),
        axis=1,
    )
)

export_metadata["page_path"] = (
    export_metadata["page_filename"]
    .map(
        lambda filename:
            portable_path(
                Path("Pages_JPG")
                / filename
            )
    )
)


# -----------------------------------------------------
# Resolve source files
# -----------------------------------------------------

export_metadata["source_crop_path"] = (
    export_metadata["local_crop_path"]
    .map(
        lambda value:
            resolve_source_path(
                value,
                PIPELINE_ROOT,
            )
    )
)

export_metadata["source_page_path"] = (
    export_metadata["local_jpg_path"]
    .map(
        lambda value:
            resolve_source_path(
                value,
                PIPELINE_ROOT,
            )
    )
)


# -----------------------------------------------------
# Validate source files
# -----------------------------------------------------

missing_crop_files = export_metadata.loc[
    export_metadata["source_crop_path"].map(
        lambda path:
            path is None
            or not path.exists()
    )
].copy()

missing_page_files = export_metadata.loc[
    export_metadata["source_page_path"].map(
        lambda path:
            path is None
            or not path.exists()
    )
].copy()

if not missing_crop_files.empty:
    examples = (
        missing_crop_files[
            [
                "crop_id",
                "local_crop_path",
            ]
        ]
        .head(20)
        .to_dict(orient="records")
    )

    raise FileNotFoundError(
        "Some retained crop image files do not exist.\n\n"
        f"Examples:\n{examples}"
    )

if not missing_page_files.empty:
    examples = (
        missing_page_files[
            [
                "page_id",
                "local_jpg_path",
            ]
        ]
        .drop_duplicates()
        .head(20)
        .to_dict(orient="records")
    )

    raise FileNotFoundError(
        "Some required page image files do not exist.\n\n"
        f"Examples:\n{examples}"
    )


# -----------------------------------------------------
# Check destination filename collisions
# -----------------------------------------------------

crop_filename_collisions = (
    export_metadata
    .groupby(
        [
            "cluster_id",
            "crop_filename",
        ]
    )["crop_id"]
    .nunique()
)

crop_filename_collisions = (
    crop_filename_collisions.loc[
        crop_filename_collisions > 1
    ]
)

if not crop_filename_collisions.empty:
    raise ValueError(
        "Different crop IDs would receive the same filename "
        "inside one cluster:\n"
        f"{crop_filename_collisions.head(20)}"
    )

page_filename_collisions = (
    export_metadata
    .groupby(
        "page_filename"
    )["page_id"]
    .nunique()
)

page_filename_collisions = (
    page_filename_collisions.loc[
        page_filename_collisions > 1
    ]
)

if not page_filename_collisions.empty:
    raise ValueError(
        "Different page IDs would receive the same filename "
        "inside Pages_JPG:\n"
        f"{page_filename_collisions.head(20)}"
    )


# -----------------------------------------------------
# Prepare output directories
# -----------------------------------------------------

if RESET_EXISTING_EXPORT:
    for generated_directory in [
        CORPUS_CLUSTERS_DIR,
        CORPUS_PAGES_DIR,
        CORPUS_METADATA_DIR,
    ]:
        if generated_directory.exists():
            shutil.rmtree(
                generated_directory
            )

CORPUS_CLUSTERS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CORPUS_PAGES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CORPUS_METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# -----------------------------------------------------
# Copy retained crop files
# -----------------------------------------------------

copied_crop_count = 0
crop_copy_errors = []

for _, row in export_metadata.iterrows():
    source_path = row["source_crop_path"]

    destination_path = (
        CORPUS_ROOT
        / row["crop_path"]
    )

    destination_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    try:
        shutil.copy2(
            source_path,
            destination_path,
        )

        copied_crop_count += 1

    except Exception as error:
        crop_copy_errors.append(
            {
                "crop_id":
                    row["crop_id"],

                "source":
                    str(source_path),

                "destination":
                    str(destination_path),

                "error":
                    str(error),
            }
        )


# -----------------------------------------------------
# Copy each required page exactly once
# -----------------------------------------------------

unique_pages = (
    export_metadata[
        [
            "page_id",
            "page_path",
            "source_page_path",
        ]
    ]
    .drop_duplicates(
        subset=[
            "page_id",
        ]
    )
    .sort_values(
        "page_id"
    )
    .reset_index(drop=True)
)

copied_page_count = 0
page_copy_errors = []

for _, row in unique_pages.iterrows():
    source_path = row["source_page_path"]

    destination_path = (
        CORPUS_ROOT
        / row["page_path"]
    )

    destination_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    try:
        shutil.copy2(
            source_path,
            destination_path,
        )

        copied_page_count += 1

    except Exception as error:
        page_copy_errors.append(
            {
                "page_id":
                    row["page_id"],

                "source":
                    str(source_path),

                "destination":
                    str(destination_path),

                "error":
                    str(error),
            }
        )


# -----------------------------------------------------
# Stop if copying failed
# -----------------------------------------------------

if crop_copy_errors or page_copy_errors:
    error_sections = []

    if crop_copy_errors:
        error_sections.append(
            "Crop copy errors:\n"
            + json.dumps(
                crop_copy_errors[:20],
                indent=2,
                ensure_ascii=False,
            )
        )

    if page_copy_errors:
        error_sections.append(
            "Page copy errors:\n"
            + json.dumps(
                page_copy_errors[:20],
                indent=2,
                ensure_ascii=False,
            )
        )

    raise RuntimeError(
        "The corpus export encountered copy errors.\n\n"
        + "\n\n".join(
            error_sections
        )
    )


# -----------------------------------------------------
# Build exact crop-level corpus metadata
# -----------------------------------------------------

export_metadata["issue_id"] = (
    export_metadata["anno_id"]
)

export_metadata["date"] = (
    pd.to_datetime(
        export_metadata["date"],
        errors="coerce",
    )
    .dt.strftime(
        "%Y-%m-%d"
    )
)

main_metadata_columns = [
    "cluster_id",
    "crop_id",
    "page_id",
    "issue_id",

    "date",
    "page_number",
    "crop_path",
    "page_path",
    "iiif_crop_url",
    "iiif_page_url",

    "periodical_title",
    "publishing_place",
    "publishing_place_gnd",
    "language",
    "mediatype",
    "attribution",
    "provider",
    "rights",
]

missing_output_columns = [
    column
    for column in main_metadata_columns
    if column not in export_metadata.columns
]

if missing_output_columns:
    raise ValueError(
        "The following required output columns could not "
        "be created:\n"
        f"{missing_output_columns}"
    )

character_corpus_metadata = (
    export_metadata[
        main_metadata_columns
    ]
    .copy()
    .sort_values(
        [
            "cluster_id",
            "date",
            "page_id",
            "crop_id",
        ],
        na_position="last",
    )
    .reset_index(drop=True)
)

character_corpus_metadata.to_csv(
    CORPUS_METADATA_PATH,
    index=False,
)


# -----------------------------------------------------
# Build cluster summary
# -----------------------------------------------------

cluster_summary_rows = []

for curated_cluster_id in curated_cluster_ids:
    corpus_cluster_id = (
        cluster_id_mapping[
            curated_cluster_id
        ]
    )

    cluster_rows = export_metadata.loc[
        export_metadata["final_cluster_id"]
        == curated_cluster_id
    ].copy()

    valid_dates = (
        pd.to_datetime(
            cluster_rows["date"],
            errors="coerce",
        )
        .dropna()
    )

    first_appearance = (
        valid_dates.min().strftime(
            "%Y-%m-%d"
        )
        if not valid_dates.empty
        else ""
    )

    last_appearance = (
        valid_dates.max().strftime(
            "%Y-%m-%d"
        )
        if not valid_dates.empty
        else ""
    )

    cluster_summary_rows.append(
        {
            "cluster_id":
                corpus_cluster_id,

            "curated_final_cluster_id":
                curated_cluster_id,

            "cluster_directory":
                portable_path(
                    Path("Clusters")
                    / corpus_cluster_id
                ),

            "crop_count":
                int(
                    cluster_rows["crop_id"]
                    .nunique()
                ),

            "page_count":
                int(
                    cluster_rows["page_id"]
                    .nunique()
                ),

            "issue_count":
                int(
                    cluster_rows["anno_id"]
                    .nunique()
                ),

            "first_appearance":
                first_appearance,

            "last_appearance":
                last_appearance,
        }
    )

cluster_summary = pd.DataFrame(
    cluster_summary_rows
)

cluster_summary.to_csv(
    CLUSTER_SUMMARY_PATH,
    index=False,
)


# -----------------------------------------------------
# Validate exported files
# -----------------------------------------------------

exported_crop_files = list(
    CORPUS_CLUSTERS_DIR.rglob(
        "*.jpg"
    )
)

exported_page_files = list(
    CORPUS_PAGES_DIR.glob(
        "*.jpg"
    )
)

metadata_crop_count = len(
    character_corpus_metadata
)

metadata_page_count = (
    character_corpus_metadata["page_id"]
    .nunique()
)

if len(exported_crop_files) != metadata_crop_count:
    raise RuntimeError(
        "Validation failed: crop-file count differs from "
        "the number of metadata rows.\n"
        f"Crop files: {len(exported_crop_files):,}\n"
        f"Metadata rows: {metadata_crop_count:,}"
    )

if len(exported_page_files) != metadata_page_count:
    raise RuntimeError(
        "Validation failed: page-file count differs from "
        "the number of unique metadata pages.\n"
        f"Page files: {len(exported_page_files):,}\n"
        f"Unique metadata pages: {metadata_page_count:,}"
    )


# -----------------------------------------------------
# Write export report
# -----------------------------------------------------

export_timestamp = (
    datetime.now()
    .isoformat(
        timespec="seconds"
    )
)

accepted_before_keep_count = len(
    accepted_before_keep
)

duplicate_alternatives_removed = (
    accepted_before_keep_count
    - len(selected_curation)
)

report_lines = [
    "=" * 75,
    "CHARACTER CORPUS EXPORT REPORT",
    "=" * 75,
    "",
    f"Export timestamp: {export_timestamp}",
    "",
    "INPUT FILES",
    "-" * 75,
    f"Final curation CSV: {FINAL_CURATION_CSV_PATH}",
    f"Crop metadata CSV: {CROP_METADATA_CSV_PATH}",
    f"Page metadata CSV: {PAGE_METADATA_CSV_PATH}",
    "",
    "OUTPUT",
    "-" * 75,
    f"Corpus root: {CORPUS_ROOT}",
    "",
    "COUNTS",
    "-" * 75,
    (
        "Accepted character crops before keep_crop filtering: "
        f"{accepted_before_keep_count:,}"
    ),
    (
        "Duplicate alternatives excluded by keep_crop: "
        f"{duplicate_alternatives_removed:,}"
    ),
    (
        "Retained crops exported: "
        f"{metadata_crop_count:,}"
    ),
    (
        "Final clusters exported: "
        f"{len(cluster_summary):,}"
    ),
    (
        "Unique pages exported: "
        f"{metadata_page_count:,}"
    ),
    "",
    "FILES",
    "-" * 75,
    f"Copied crop files: {copied_crop_count:,}",
    f"Copied page files: {copied_page_count:,}",
    f"Crop copy errors: {len(crop_copy_errors):,}",
    f"Page copy errors: {len(page_copy_errors):,}",
    "",
    "METADATA",
    "-" * 75,
    "Metadata/character_corpus_metadata.csv",
    "Metadata/cluster_summary.csv",
    "",
    "VALIDATION",
    "-" * 75,
    "One metadata row per retained crop: PASSED",
    "One crop file per metadata row: PASSED",
    "One page file per unique page_id: PASSED",
    "Maximum one retained crop per cluster/page: PASSED",
    "Only corpus-relative paths stored in metadata: PASSED",
    "Exact requested metadata schema: PASSED",
    "",
    "HTML FRONTEND",
    "-" * 75,
    "Not generated by this cell.",
    "Run the separate HTML frontend cell afterward.",
    "",
    "=" * 75,
]

EXPORT_REPORT_PATH.write_text(
    "\n".join(report_lines),
    encoding="utf-8",
)


# -----------------------------------------------------
# Print summary
# -----------------------------------------------------

print("=" * 75)
print("CHARACTER CORPUS EXPORT COMPLETE")
print("=" * 75)

print(
    "Corpus root:",
    CORPUS_ROOT,
)

print()

print(
    "Final clusters:",
    f"{len(cluster_summary):,}",
)

print(
    "Retained crops:",
    f"{metadata_crop_count:,}",
)

print(
    "Unique pages:",
    f"{metadata_page_count:,}",
)

print(
    "Duplicate alternatives removed:",
    f"{duplicate_alternatives_removed:,}",
)

print()

print("Corpus metadata:")
print(CORPUS_METADATA_PATH)

print()

print("Cluster summary:")
print(CLUSTER_SUMMARY_PATH)

print()

print("Export report:")
print(EXPORT_REPORT_PATH)

print()

print(
    "The HTML frontend has not been generated yet."
)

print("=" * 75)


# -----------------------------------------------------
# Display previews
# -----------------------------------------------------

print()
print("CHARACTER CORPUS METADATA PREVIEW")

display(
    character_corpus_metadata.head(10)
)

print()
print("CLUSTER SUMMARY PREVIEW")

display(
    cluster_summary.head(10)
)

##8) Visualisation of results

Two plots are created from the results of the corpus creation:

 1) One showing the appearance of the characters per year over the fifty year publication span of the periodical

 2) One showing the appearances of the fice most prominent characters (prominence is based on number of appearance)

In [ ]:
# =====================================================
# Visualize character appearances over publication time
# =====================================================

# -----------------------------------------------------
# Paths
# -----------------------------------------------------

PIPELINE_ROOT = Path(PIPELINE_ROOT)

CORPUS_METADATA_PATH = (
    PIPELINE_ROOT
    / "Corpus"
    / "Metadata"
    / "character_corpus_metadata.csv"
)

VISUALIZATION_OUTPUT_DIR = (
    PIPELINE_ROOT
    / "Corpus"
    / "Visualizations"
)

VISUALIZATION_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# -----------------------------------------------------
# Load final corpus metadata
# -----------------------------------------------------

if not CORPUS_METADATA_PATH.exists():
    raise FileNotFoundError(
        "Final corpus metadata not found:\n"
        f"{CORPUS_METADATA_PATH}"
    )

corpus_metadata = pd.read_csv(
    CORPUS_METADATA_PATH,
    dtype=str,
)

corpus_metadata.columns = (
    corpus_metadata.columns
    .astype(str)
    .str.strip()
)


# -----------------------------------------------------
# Validate required columns
# -----------------------------------------------------

required_columns = {
    "cluster_id",
    "date",
}

missing_columns = (
    required_columns
    - set(corpus_metadata.columns)
)

if missing_columns:
    raise ValueError(
        "The corpus metadata is missing required columns:\n"
        f"{sorted(missing_columns)}"
    )


# -----------------------------------------------------
# Normalize and parse dates
# -----------------------------------------------------

corpus_metadata["cluster_id"] = (
    corpus_metadata["cluster_id"]
    .fillna("")
    .astype(str)
    .str.strip()
)

corpus_metadata["date"] = pd.to_datetime(
    corpus_metadata["date"],
    errors="coerce",
)

missing_dates = corpus_metadata[
    corpus_metadata["date"].isna()
]

if not missing_dates.empty:
    raise ValueError(
        "Some corpus rows contain invalid or missing dates.\n\n"
        "Example rows:\n"
        f"{missing_dates.head(10)}"
    )

corpus_metadata["year"] = (
    corpus_metadata["date"].dt.year
)


# -----------------------------------------------------
# Fixed publication-year range
# -----------------------------------------------------

PUBLICATION_START_YEAR = 1871
PUBLICATION_END_YEAR = 1925

all_years = pd.Index(
    range(
        PUBLICATION_START_YEAR,
        PUBLICATION_END_YEAR + 1,
    ),
    name="year",
)


# -----------------------------------------------------
# Validate corpus dates against publication range
# -----------------------------------------------------

outside_publication_range = corpus_metadata.loc[
    (
        corpus_metadata["year"]
        < PUBLICATION_START_YEAR
    )
    | (
        corpus_metadata["year"]
        > PUBLICATION_END_YEAR
    )
]

if not outside_publication_range.empty:
    raise ValueError(
        "Some corpus rows fall outside the expected "
        "publication range 1871–1925.\n\n"
        "Example rows:\n"
        f"{outside_publication_range.head(10)}"
    )


print("=" * 80)
print("CHARACTER APPEARANCE VISUALIZATION")
print("=" * 80)

print(
    "Publication period:",
    f"{PUBLICATION_START_YEAR}–{PUBLICATION_END_YEAR}",
)

print(
    "Character appearances:",
    f"{len(corpus_metadata):,}",
)

print(
    "Character clusters:",
    f"{corpus_metadata['cluster_id'].nunique():,}",
)

print("=" * 80)


# =====================================================
# 1. ALL CHARACTER APPEARANCES PER YEAR
# =====================================================

annual_total = (
    corpus_metadata
    .groupby("year")
    .size()
    .reindex(
        all_years,
        fill_value=0,
    )
)


# -----------------------------------------------------
# Bar chart
# -----------------------------------------------------

fig, ax = plt.subplots(
    figsize=(16, 7)
)

ax.bar(
    annual_total.index,
    annual_total.values,
    width=0.8,
)

ax.set_title(
    "Character appearances per year"
)

ax.set_xlabel(
    "Year"
)

ax.set_ylabel(
    "Number of character appearances"
)

ax.set_xlim(
    PUBLICATION_START_YEAR - 0.7,
    PUBLICATION_END_YEAR + 0.7,
)

# Show every fifth year as a major x-axis label.
ax.set_xticks(
    range(
        PUBLICATION_START_YEAR,
        PUBLICATION_END_YEAR + 1,
        5,
    )
)

ax.grid(
    axis="y",
    alpha=0.3,
)

fig.tight_layout()


total_output_path = (
    VISUALIZATION_OUTPUT_DIR
    / "character_appearances_per_year.png"
)

fig.savefig(
    total_output_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print()
print(
    "Saved total-appearance visualization:"
)

print(total_output_path)


# =====================================================
# 2. IDENTIFY THE FIVE MOST PROMINENT CHARACTERS
# =====================================================

character_totals = (
    corpus_metadata
    .groupby("cluster_id")
    .size()
    .sort_values(
        ascending=False
    )
)

top_five_clusters = (
    character_totals
    .head(5)
    .index
    .tolist()
)


print()
print("=" * 80)
print("FIVE MOST PROMINENT CHARACTERS")
print("=" * 80)

for rank, cluster_id in enumerate(
    top_five_clusters,
    start=1,
):
    print(
        f"{rank}. {cluster_id}: "
        f"{character_totals.loc[cluster_id]:,} appearances"
    )

print("=" * 80)


# -----------------------------------------------------
# Annual counts for top five
# -----------------------------------------------------

top_five_annual = (
    corpus_metadata.loc[
        corpus_metadata["cluster_id"].isin(
            top_five_clusters
        )
    ]
    .groupby(
        [
            "year",
            "cluster_id",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(
        all_years,
        fill_value=0,
    )
)

# Keep columns ordered by overall prominence.
top_five_annual = (
    top_five_annual[
        top_five_clusters
    ]
)


# =====================================================
# 3. TOP FIVE CHARACTERS PER YEAR
# =====================================================

fig, ax = plt.subplots(
    figsize=(16, 7)
)

for cluster_id in top_five_clusters:
    ax.plot(
        top_five_annual.index,
        top_five_annual[cluster_id],
        linewidth=2,
        marker="o",
        markersize=3,
        label=cluster_id,
    )

ax.set_title(
    "Annual appearances of the five most prominent characters"
)

ax.set_xlabel(
    "Year"
)

ax.set_ylabel(
    "Number of character appearances"
)

ax.set_xlim(
    PUBLICATION_START_YEAR,
    PUBLICATION_END_YEAR,
)

ax.set_xticks(
    range(
        PUBLICATION_START_YEAR,
        PUBLICATION_END_YEAR + 1,
        5,
    )
)

ax.grid(
    axis="y",
    alpha=0.3,
)

ax.legend(
    title="Character cluster",
    frameon=False,
)

fig.tight_layout()


top_five_output_path = (
    VISUALIZATION_OUTPUT_DIR
    / "top_five_character_appearances_per_year.png"
)

fig.savefig(
    top_five_output_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print()
print(
    "Saved top-five visualization:"
)

print(top_five_output_path)


# =====================================================
# 4. SAVE AGGREGATED DATA
# =====================================================

annual_total_output = (
    annual_total
    .rename("appearances")
    .reset_index()
)

annual_total_csv_path = (
    VISUALIZATION_OUTPUT_DIR
    / "character_appearances_per_year.csv"
)

annual_total_output.to_csv(
    annual_total_csv_path,
    index=False,
)


top_five_output = (
    top_five_annual
    .reset_index()
)

top_five_csv_path = (
    VISUALIZATION_OUTPUT_DIR
    / "top_five_character_appearances_per_year.csv"
)

top_five_output.to_csv(
    top_five_csv_path,
    index=False,
)


print()
print("=" * 80)
print("VISUALIZATION EXPORT COMPLETE")
print("=" * 80)

print(
    "Annual totals:",
    annual_total_csv_path,
)

print(
    "Top-five annual counts:",
    top_five_csv_path,
)

print("=" * 80)

##9) Creation of a HTML Frontend View

To make the corpus more accesible, a simple HTML frontend is created, that shows each cluster with a representative crop and the possibility to show access all pages and crops of this cluster.

 Reads:

 Corpus/Metadata/character_corpus_metadata.csv
 Corpus/Metadata/cluster_summary.csv

 Creates:

 Corpus/HTML_frontend/
 ├── index.html
 └── clusters/
     ├── cluster_0001.html
     ├── cluster_0002.html
     └── ...

 Frontend behavior:

 INDEX PAGE
 - Displays one card per character cluster.
 - Clicking a cluster card opens the individual cluster page.

 CLUSTER PAGE
 - Displays every retained crop in chronological order.
 - Single-clicking a crop opens it in a larger lightbox.
 - The lightbox uses a nearly opaque black background.
 - Left and right arrow buttons move through the cluster.
 - Keyboard left/right arrows also move through the cluster.
 - Escape or clicking the dark background closes the lightbox.
 - Double-clicking the enlarged crop opens its page-level
   IIIF URL in a new browser tab.

 Only crop images are loaded by the HTML page.
 Newspaper pages are requested through IIIF only after
 double-clicking the enlarged crop.

In [ ]:
# =====================================================
# Create static HTML frontend with cluster lightbox
# =====================================================

RESET_EXISTING_HTML = True


# -----------------------------------------------------
# Paths
# -----------------------------------------------------

CORPUS_METADATA_PATH = (
    CORPUS_ROOT
    / "Metadata"
    / "character_corpus_metadata.csv"
)

CLUSTER_SUMMARY_PATH = (
    CORPUS_ROOT
    / "Metadata"
    / "cluster_summary.csv"
)

HTML_FRONTEND_DIR = (
    CORPUS_ROOT
    / "HTML_frontend"
)

HTML_CLUSTER_PAGES_DIR = (
    HTML_FRONTEND_DIR
    / "clusters"
)

HTML_INDEX_PATH = (
    HTML_FRONTEND_DIR
    / "index.html"
)


# -----------------------------------------------------
# Helper functions
# -----------------------------------------------------

def normalize_html_text(value):
    """
    Convert a metadata value into a clean string.
    """
    if pd.isna(value):
        return ""

    return str(value).strip()


def make_relative_browser_path(
    corpus_relative_path,
    levels_up,
):
    """
    Convert a Corpus-relative path into a relative browser path
    from an HTML file located below Corpus/.

    index.html:
        Corpus/HTML_frontend/index.html
        -> ../Clusters/...

    cluster page:
        Corpus/HTML_frontend/clusters/cluster_0001.html
        -> ../../Clusters/...
    """
    prefix = "../" * levels_up

    normalized_path = (
        str(corpus_relative_path)
        .replace("\\", "/")
        .lstrip("/")
    )

    return prefix + normalized_path


def json_for_script(value):
    """
    Serialize Python data safely for insertion into JavaScript.
    """
    return json.dumps(
        value,
        ensure_ascii=False,
        separators=(",", ":"),
    )


# -----------------------------------------------------
# Validate input files
# -----------------------------------------------------

if not CORPUS_METADATA_PATH.exists():
    raise FileNotFoundError(
        "Corpus metadata was not found:\n"
        f"{CORPUS_METADATA_PATH}\n\n"
        "Run the corpus export cell first."
    )

if not CLUSTER_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        "Cluster summary was not found:\n"
        f"{CLUSTER_SUMMARY_PATH}\n\n"
        "Run the corpus export cell first."
    )


# -----------------------------------------------------
# Load corpus metadata
# -----------------------------------------------------

corpus_metadata = pd.read_csv(
    CORPUS_METADATA_PATH,
    dtype=str,
)

cluster_summary = pd.read_csv(
    CLUSTER_SUMMARY_PATH,
    dtype=str,
)

for dataframe in [
    corpus_metadata,
    cluster_summary,
]:
    dataframe.columns = (
        dataframe.columns
        .astype(str)
        .str.strip()
    )

    for column in dataframe.columns:
        dataframe[column] = (
            dataframe[column]
            .map(normalize_html_text)
        )


# -----------------------------------------------------
# Validate required columns
# -----------------------------------------------------

required_metadata_columns = {
    "cluster_id",
    "crop_id",
    "page_id",
    "issue_id",
    "date",
    "page_number",
    "crop_path",
    "page_path",
    "iiif_crop_url",
    "iiif_page_url",
}

required_summary_columns = {
    "cluster_id",
    "crop_count",
    "page_count",
    "issue_count",
    "first_appearance",
    "last_appearance",
}

missing_metadata_columns = (
    required_metadata_columns
    - set(corpus_metadata.columns)
)

missing_summary_columns = (
    required_summary_columns
    - set(cluster_summary.columns)
)

if missing_metadata_columns:
    raise ValueError(
        "Corpus metadata is missing columns required by "
        "the frontend:\n"
        f"{sorted(missing_metadata_columns)}"
    )

if missing_summary_columns:
    raise ValueError(
        "Cluster summary is missing columns required by "
        "the frontend:\n"
        f"{sorted(missing_summary_columns)}"
    )


# -----------------------------------------------------
# Validate identifiers
# -----------------------------------------------------

if corpus_metadata["crop_id"].duplicated().any():
    duplicated_crop_ids = (
        corpus_metadata.loc[
            corpus_metadata[
                "crop_id"
            ].duplicated(keep=False),
            "crop_id",
        ]
        .drop_duplicates()
        .tolist()
    )

    raise ValueError(
        "Corpus metadata contains duplicated crop_id values:\n"
        f"{duplicated_crop_ids[:20]}"
    )

if cluster_summary["cluster_id"].duplicated().any():
    duplicated_cluster_ids = (
        cluster_summary.loc[
            cluster_summary[
                "cluster_id"
            ].duplicated(keep=False),
            "cluster_id",
        ]
        .drop_duplicates()
        .tolist()
    )

    raise ValueError(
        "Cluster summary contains duplicated cluster_id values:\n"
        f"{duplicated_cluster_ids[:20]}"
    )


# -----------------------------------------------------
# Validate cluster relationships
# -----------------------------------------------------

metadata_cluster_ids = set(
    corpus_metadata["cluster_id"]
)

summary_cluster_ids = set(
    cluster_summary["cluster_id"]
)

missing_from_summary = sorted(
    metadata_cluster_ids
    - summary_cluster_ids
)

missing_from_metadata = sorted(
    summary_cluster_ids
    - metadata_cluster_ids
)

if missing_from_summary:
    raise ValueError(
        "Some corpus-metadata clusters are absent from "
        "cluster_summary.csv:\n"
        f"{missing_from_summary[:20]}"
    )

if missing_from_metadata:
    raise ValueError(
        "Some summary clusters contain no corpus metadata rows:\n"
        f"{missing_from_metadata[:20]}"
    )


# -----------------------------------------------------
# Validate local crop files
# -----------------------------------------------------

missing_local_crops = []

for row in corpus_metadata.itertuples(
    index=False
):
    crop_file_path = (
        CORPUS_ROOT
        / row.crop_path
    )

    if not crop_file_path.exists():
        missing_local_crops.append(
            {
                "crop_id":
                    row.crop_id,

                "crop_path":
                    row.crop_path,
            }
        )

if missing_local_crops:
    raise FileNotFoundError(
        "Some crop files referenced by corpus metadata "
        "do not exist:\n"
        f"{missing_local_crops[:20]}"
    )


# -----------------------------------------------------
# Prepare frontend directory
# -----------------------------------------------------

if (
    RESET_EXISTING_HTML
    and HTML_FRONTEND_DIR.exists()
):
    shutil.rmtree(
        HTML_FRONTEND_DIR
    )

HTML_CLUSTER_PAGES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# -----------------------------------------------------
# Shared CSS
# -----------------------------------------------------

shared_css = """
* {
    box-sizing: border-box;
}

html {
    scroll-behavior: smooth;
}

body {
    margin: 0;
    font-family: Arial, Helvetica, sans-serif;
    background: #f3f4f6;
    color: #111827;
}

body.lightbox-open {
    overflow: hidden;
}

header {
    padding: 24px;
    background: #111827;
    color: white;
}

header h1 {
    margin: 0 0 8px;
}

header p {
    margin: 5px 0;
}

main {
    padding: 24px;
}

a {
    color: #1d4ed8;
}

.top-link {
    display: inline-block;
    margin-bottom: 18px;
    font-weight: bold;
    text-decoration: none;
}

.top-link:hover {
    text-decoration: underline;
}

.cluster-grid,
.crop-grid {
    display: grid;
    gap: 18px;
}

.cluster-grid {
    grid-template-columns:
        repeat(
            auto-fill,
            minmax(260px, 1fr)
        );
}

.crop-grid {
    grid-template-columns:
        repeat(
            auto-fill,
            minmax(240px, 1fr)
        );
}

.cluster-card,
.crop-card {
    overflow: hidden;
    border: 1px solid #d1d5db;
    border-radius: 10px;
    background: white;
    transition:
        transform 0.15s ease,
        box-shadow 0.15s ease;
}

.cluster-card {
    cursor: pointer;
}

.cluster-card:hover {
    transform: translateY(-3px);
    box-shadow:
        0 8px 20px
        rgba(0, 0, 0, 0.14);
}

.crop-card {
    cursor: pointer;
}

.crop-card:hover {
    transform: translateY(-2px);
    box-shadow:
        0 6px 16px
        rgba(0, 0, 0, 0.14);
}

.cluster-card img,
.crop-card img {
    display: block;
    width: 100%;
    height: 260px;
    object-fit: contain;
    background: #f9fafb;
    user-select: none;
}

.card-info {
    padding: 14px;
}

.card-info h2,
.card-info h3 {
    margin: 0 0 10px;
    overflow-wrap: anywhere;
}

.card-info p {
    margin: 5px 0;
    overflow-wrap: anywhere;
}

.cluster-card-link {
    color: inherit;
    text-decoration: none;
}

.cluster-card-link:focus-visible,
.crop-card:focus-visible {
    outline: 4px solid #2563eb;
    outline-offset: 3px;
}

.summary-box {
    margin-bottom: 20px;
    padding: 16px;
    border: 1px solid #d1d5db;
    border-radius: 10px;
    background: white;
}

.summary-box p {
    margin: 6px 0;
}

.interaction-note {
    margin-top: 12px !important;
    color: #4b5563;
    font-size: 13px;
    font-style: italic;
}

.metadata-line {
    font-size: 14px;
}

.page-instruction {
    margin-top: 10px;
    color: #d1d5db;
    font-size: 14px;
}


/* --------------------------------------------------
   Lightbox
   -------------------------------------------------- */

.lightbox {
    position: fixed;
    inset: 0;
    z-index: 1000;
    display: none;
    align-items: center;
    justify-content: center;
    padding: 30px 90px;
    background: rgba(0, 0, 0, 0.88);
    backdrop-filter: blur(2px);
}

.lightbox.open {
    display: flex;
}

.lightbox-content {
    position: relative;
    display: flex;
    flex-direction: column;
    align-items: center;
    max-width: 100%;
    max-height: 100%;
}

.lightbox-image-wrapper {
    display: flex;
    align-items: center;
    justify-content: center;
    max-width: 100%;
    max-height: calc(100vh - 190px);
}

.lightbox-image {
    display: block;
    max-width: min(82vw, 1400px);
    max-height: calc(100vh - 190px);
    object-fit: contain;
    border-radius: 8px;
    background: white;
    box-shadow:
        0 18px 50px
        rgba(0, 0, 0, 0.45);
    user-select: none;
    cursor: zoom-in;
}

.lightbox-image.has-iiif-page {
    cursor: pointer;
}

.lightbox-caption {
    width: min(82vw, 1400px);
    margin-top: 14px;
    padding: 12px 16px;
    border-radius: 8px;
    background: rgba(17, 24, 39, 0.92);
    color: white;
    text-align: center;
}

.lightbox-caption p {
    margin: 4px 0;
}

.lightbox-instruction {
    color: #d1d5db;
    font-size: 13px;
    font-style: italic;
}

.lightbox-close,
.lightbox-arrow {
    position: fixed;
    z-index: 1001;
    border: none;
    color: white;
    background: rgba(17, 24, 39, 0.7);
    cursor: pointer;
    transition:
        background 0.15s ease,
        transform 0.15s ease;
}

.lightbox-close:hover,
.lightbox-arrow:hover {
    background: rgba(17, 24, 39, 0.95);
}

.lightbox-close:focus-visible,
.lightbox-arrow:focus-visible {
    outline: 4px solid white;
    outline-offset: 3px;
}

.lightbox-close {
    top: 20px;
    right: 24px;
    width: 48px;
    height: 48px;
    border-radius: 999px;
    font-size: 28px;
    line-height: 1;
}

.lightbox-arrow {
    top: 50%;
    width: 58px;
    height: 76px;
    border-radius: 10px;
    transform: translateY(-50%);
    font-size: 42px;
    line-height: 1;
}

.lightbox-arrow:hover {
    transform:
        translateY(-50%)
        scale(1.05);
}

.lightbox-arrow.previous {
    left: 18px;
}

.lightbox-arrow.next {
    right: 18px;
}

.lightbox-counter {
    position: fixed;
    top: 24px;
    left: 50%;
    z-index: 1001;
    padding: 8px 13px;
    border-radius: 999px;
    transform: translateX(-50%);
    background: rgba(17, 24, 39, 0.75);
    color: white;
    font-weight: bold;
}

@media (max-width: 720px) {
    .lightbox {
        padding:
            70px
            54px
            24px;
    }

    .lightbox-image {
        max-width: calc(100vw - 110px);
        max-height: calc(100vh - 220px);
    }

    .lightbox-caption {
        width: calc(100vw - 110px);
    }

    .lightbox-arrow {
        width: 42px;
        height: 62px;
        font-size: 32px;
    }

    .lightbox-arrow.previous {
        left: 6px;
    }

    .lightbox-arrow.next {
        right: 6px;
    }

    .lightbox-close {
        top: 12px;
        right: 12px;
    }
}
"""


# -----------------------------------------------------
# Shared lightbox JavaScript
# -----------------------------------------------------

lightbox_javascript = """
let lightboxItems = [];
let currentLightboxIndex = 0;


function initializeLightbox(items) {
    lightboxItems = Array.isArray(items)
        ? items
        : [];
}


function openLightbox(index) {
    if (
        lightboxItems.length === 0
        || index < 0
        || index >= lightboxItems.length
    ) {
        return;
    }

    currentLightboxIndex = index;

    const lightbox =
        document.getElementById(
            "cluster-lightbox"
        );

    lightbox.classList.add("open");

    lightbox.setAttribute(
        "aria-hidden",
        "false"
    );

    document.body.classList.add(
        "lightbox-open"
    );

    renderLightboxItem();
}


function closeLightbox() {
    const lightbox =
        document.getElementById(
            "cluster-lightbox"
        );

    lightbox.classList.remove("open");

    lightbox.setAttribute(
        "aria-hidden",
        "true"
    );

    document.body.classList.remove(
        "lightbox-open"
    );
}


function moveLightbox(direction) {
    if (lightboxItems.length === 0) {
        return;
    }

    currentLightboxIndex =
        (
            currentLightboxIndex
            + direction
            + lightboxItems.length
        )
        % lightboxItems.length;

    renderLightboxItem();
}


function renderLightboxItem() {
    const item =
        lightboxItems[
            currentLightboxIndex
        ];

    const image =
        document.getElementById(
            "lightbox-image"
        );

    const cropId =
        document.getElementById(
            "lightbox-crop-id"
        );

    const date =
        document.getElementById(
            "lightbox-date"
        );

    const pageId =
        document.getElementById(
            "lightbox-page-id"
        );

    const pageNumber =
        document.getElementById(
            "lightbox-page-number"
        );

    const instruction =
        document.getElementById(
            "lightbox-instruction"
        );

    const counter =
        document.getElementById(
            "lightbox-counter"
        );

    image.src =
        item.cropImagePath;

    image.alt =
        item.cropId;

    image.dataset.iiifPageUrl =
        item.iiifPageUrl || "";

    image.classList.toggle(
        "has-iiif-page",
        Boolean(item.iiifPageUrl)
    );

    cropId.textContent =
        item.cropId || "";

    date.textContent =
        item.date || "";

    pageId.textContent =
        item.pageId || "";

    pageNumber.textContent =
        item.pageNumber || "";

    counter.textContent =
        (
            currentLightboxIndex + 1
        )
        + " / "
        + lightboxItems.length;

    if (item.iiifPageUrl) {
        instruction.textContent =
            "Double-click the enlarged crop "
            + "to open its source page through IIIF.";
    } else {
        instruction.textContent =
            "No page-level IIIF URL is available.";
    }
}


function openCurrentIiifPage() {
    if (lightboxItems.length === 0) {
        return;
    }

    const item =
        lightboxItems[
            currentLightboxIndex
        ];

    if (!item.iiifPageUrl) {
        return;
    }

    window.open(
        item.iiifPageUrl,
        "_blank",
        "noopener,noreferrer"
    );
}


function handleLightboxBackdropClick(event) {
    if (
        event.target.id
        === "cluster-lightbox"
    ) {
        closeLightbox();
    }
}


document.addEventListener(
    "keydown",
    event => {
        const lightbox =
            document.getElementById(
                "cluster-lightbox"
            );

        if (
            !lightbox
            || !lightbox.classList.contains(
                "open"
            )
        ) {
            return;
        }

        if (event.key === "Escape") {
            event.preventDefault();
            closeLightbox();
        }

        if (event.key === "ArrowLeft") {
            event.preventDefault();
            moveLightbox(-1);
        }

        if (event.key === "ArrowRight") {
            event.preventDefault();
            moveLightbox(1);
        }
    }
);
"""


# -----------------------------------------------------
# Generate one page for every cluster
# -----------------------------------------------------

cluster_page_links = {}

cluster_summary = (
    cluster_summary
    .sort_values(
        "cluster_id"
    )
    .reset_index(drop=True)
)

for summary_row in cluster_summary.itertuples(
    index=False
):
    cluster_id = (
        summary_row.cluster_id
    )

    cluster_rows = (
        corpus_metadata.loc[
            corpus_metadata["cluster_id"]
            == cluster_id
        ]
        .copy()
        .sort_values(
            [
                "date",
                "page_id",
                "crop_id",
            ],
            na_position="last",
        )
        .reset_index(drop=True)
    )

    if cluster_rows.empty:
        raise ValueError(
            f"Cluster {cluster_id} has no crop metadata."
        )

    crop_cards = []
    lightbox_items = []

    for crop_position, crop_row in enumerate(
        cluster_rows.itertuples(
            index=False
        )
    ):
        crop_image_path = (
            make_relative_browser_path(
                crop_row.crop_path,
                levels_up=2,
            )
        )

        lightbox_items.append(
            {
                "cropImagePath":
                    crop_image_path,

                "cropId":
                    crop_row.crop_id,

                "date":
                    crop_row.date,

                "pageId":
                    crop_row.page_id,

                "pageNumber":
                    crop_row.page_number,

                "iiifPageUrl":
                    crop_row.iiif_page_url,
            }
        )

        iiif_available = bool(
            crop_row.iiif_page_url
        )

        iiif_note = (
            "Open crop viewer"
            if iiif_available
            else "Open crop viewer; no IIIF page available"
        )

        crop_cards.append(
            f"""
            <article
                class="crop-card"
                tabindex="0"
                role="button"
                aria-label="{escape(iiif_note, quote=True)}"
                onclick="openLightbox({crop_position})"
                onkeydown="
                    if (
                        event.key === 'Enter'
                        || event.key === ' '
                    ) {{
                        event.preventDefault();
                        openLightbox({crop_position});
                    }}
                "
            >
                <img
                    src="{escape(crop_image_path, quote=True)}"
                    alt="{escape(crop_row.crop_id, quote=True)}"
                    loading="lazy"
                    draggable="false"
                >

                <div class="card-info">
                    <h3>
                        {escape(crop_row.crop_id)}
                    </h3>

                    <p class="metadata-line">
                        <strong>Date:</strong>
                        {escape(crop_row.date)}
                    </p>

                    <p class="metadata-line">
                        <strong>Page ID:</strong>
                        {escape(crop_row.page_id)}
                    </p>

                    <p class="metadata-line">
                        <strong>Issue ID:</strong>
                        {escape(crop_row.issue_id)}
                    </p>

                    <p class="metadata-line">
                        <strong>Page number:</strong>
                        {escape(crop_row.page_number)}
                    </p>

                    <p class="interaction-note">
                        Click to enlarge.
                    </p>
                </div>
            </article>
            """
        )

    lightbox_items_json = (
        json_for_script(
            lightbox_items
        )
    )

    cluster_document = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">

    <meta
        name="viewport"
        content="width=device-width, initial-scale=1.0"
    >

    <title>
        {escape(cluster_id)}
    </title>

    <style>
        {shared_css}
    </style>
</head>

<body>
    <header>
        <h1>
            {escape(cluster_id)}
        </h1>

        <p>
            {escape(str(summary_row.crop_count))}
            retained crops
        </p>

        <p class="page-instruction">
            Click a crop to enlarge it. Use the arrow
            buttons or keyboard arrow keys to move through
            the cluster. Double-click the enlarged crop to
            open its source page through IIIF.
        </p>
    </header>

    <main>
        <a
            class="top-link"
            href="../index.html"
        >
            ← Back to all clusters
        </a>

        <section class="summary-box">
            <p>
                <strong>Crops:</strong>
                {escape(str(summary_row.crop_count))}
            </p>

            <p>
                <strong>Pages:</strong>
                {escape(str(summary_row.page_count))}
            </p>

            <p>
                <strong>Issues:</strong>
                {escape(str(summary_row.issue_count))}
            </p>

            <p>
                <strong>First appearance:</strong>
                {escape(str(summary_row.first_appearance))}
            </p>

            <p>
                <strong>Last appearance:</strong>
                {escape(str(summary_row.last_appearance))}
            </p>
        </section>

        <section class="crop-grid">
            {''.join(crop_cards)}
        </section>
    </main>


    <!-- Lightbox -->
    <div
        id="cluster-lightbox"
        class="lightbox"
        role="dialog"
        aria-modal="true"
        aria-hidden="true"
        aria-label="Cluster crop viewer"
        onclick="handleLightboxBackdropClick(event)"
    >
        <div
            id="lightbox-counter"
            class="lightbox-counter"
        ></div>

        <button
            class="lightbox-close"
            type="button"
            aria-label="Close crop viewer"
            onclick="closeLightbox()"
        >
            ×
        </button>

        <button
            class="lightbox-arrow previous"
            type="button"
            aria-label="Previous crop"
            onclick="
                event.stopPropagation();
                moveLightbox(-1);
            "
        >
            ‹
        </button>

        <div
            class="lightbox-content"
            onclick="event.stopPropagation()"
        >
            <div class="lightbox-image-wrapper">
                <img
                    id="lightbox-image"
                    class="lightbox-image"
                    src=""
                    alt=""
                    draggable="false"
                    ondblclick="
                        event.stopPropagation();
                        openCurrentIiifPage();
                    "
                >
            </div>

            <div class="lightbox-caption">
                <p>
                    <strong id="lightbox-crop-id"></strong>
                </p>

                <p>
                    Date:
                    <span id="lightbox-date"></span>
                </p>

                <p>
                    Page ID:
                    <span id="lightbox-page-id"></span>
                </p>

                <p>
                    Page number:
                    <span id="lightbox-page-number"></span>
                </p>

                <p
                    id="lightbox-instruction"
                    class="lightbox-instruction"
                ></p>
            </div>
        </div>

        <button
            class="lightbox-arrow next"
            type="button"
            aria-label="Next crop"
            onclick="
                event.stopPropagation();
                moveLightbox(1);
            "
        >
            ›
        </button>
    </div>


    <script>
        {lightbox_javascript}

        initializeLightbox(
            {lightbox_items_json}
        );
    </script>
</body>
</html>
"""

    cluster_page_path = (
        HTML_CLUSTER_PAGES_DIR
        / f"{cluster_id}.html"
    )

    cluster_page_path.write_text(
        cluster_document,
        encoding="utf-8",
    )

    cluster_page_links[
        cluster_id
    ] = (
        "clusters/"
        + cluster_page_path.name
    )


# -----------------------------------------------------
# Generate main index
# -----------------------------------------------------

cluster_cards = []

for summary_row in cluster_summary.itertuples(
    index=False
):
    cluster_id = (
        summary_row.cluster_id
    )

    cluster_rows = (
        corpus_metadata.loc[
            corpus_metadata["cluster_id"]
            == cluster_id
        ]
        .copy()
        .sort_values(
            [
                "date",
                "page_id",
                "crop_id",
            ],
            na_position="last",
        )
        .reset_index(drop=True)
    )

    representative_row = (
        cluster_rows.iloc[0]
    )

    representative_crop_path = (
        make_relative_browser_path(
            representative_row[
                "crop_path"
            ],
            levels_up=1,
        )
    )

    cluster_page_link = (
        cluster_page_links[
            cluster_id
        ]
    )

    cluster_cards.append(
        f"""
        <a
            class="cluster-card-link"
            href="{escape(cluster_page_link, quote=True)}"
        >
            <article class="cluster-card">
                <img
                    src="{escape(
                        representative_crop_path,
                        quote=True,
                    )}"
                    alt="{escape(cluster_id, quote=True)}"
                    loading="lazy"
                    draggable="false"
                >

                <div class="card-info">
                    <h2>
                        {escape(cluster_id)}
                    </h2>

                    <p>
                        <strong>Crops:</strong>
                        {escape(str(summary_row.crop_count))}
                    </p>

                    <p>
                        <strong>Pages:</strong>
                        {escape(str(summary_row.page_count))}
                    </p>

                    <p>
                        <strong>Issues:</strong>
                        {escape(str(summary_row.issue_count))}
                    </p>

                    <p>
                        <strong>First appearance:</strong>
                        {escape(
                            str(
                                summary_row.first_appearance
                            )
                        )}
                    </p>

                    <p>
                        <strong>Last appearance:</strong>
                        {escape(
                            str(
                                summary_row.last_appearance
                            )
                        )}
                    </p>

                    <p class="interaction-note">
                        Click to inspect this cluster.
                    </p>
                </div>
            </article>
        </a>
        """
    )

index_document = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">

    <meta
        name="viewport"
        content="width=device-width, initial-scale=1.0"
    >

    <title>
        Character Corpus
    </title>

    <style>
        {shared_css}
    </style>
</head>

<body>
    <header>
        <h1>
            Character Corpus
        </h1>

        <p>
            {len(cluster_summary):,}
            character clusters
        </p>

        <p>
            {len(corpus_metadata):,}
            retained crops
        </p>

        <p>
            {
                corpus_metadata[
                    "page_id"
                ].nunique()
            :,}
            unique pages
        </p>

        <p class="page-instruction">
            Click a cluster card to inspect all crops
            assigned to that character.
        </p>
    </header>

    <main>
        <section class="cluster-grid">
            {''.join(cluster_cards)}
        </section>
    </main>
</body>
</html>
"""

HTML_INDEX_PATH.write_text(
    index_document,
    encoding="utf-8",
)


# -----------------------------------------------------
# Validate generated frontend
# -----------------------------------------------------

generated_cluster_pages = list(
    HTML_CLUSTER_PAGES_DIR.glob(
        "cluster_*.html"
    )
)

if (
    len(generated_cluster_pages)
    != len(cluster_summary)
):
    raise RuntimeError(
        "Frontend validation failed: the number of "
        "cluster HTML pages differs from the cluster "
        "summary count.\n"
        f"HTML pages: {len(generated_cluster_pages):,}\n"
        f"Clusters: {len(cluster_summary):,}"
    )

if not HTML_INDEX_PATH.exists():
    raise RuntimeError(
        "Frontend validation failed: index.html "
        "was not created."
    )

missing_cluster_page_files = [
    cluster_id
    for cluster_id, relative_link
    in cluster_page_links.items()
    if not (
        HTML_FRONTEND_DIR
        / relative_link
    ).exists()
]

if missing_cluster_page_files:
    raise RuntimeError(
        "Frontend validation failed: some cluster pages "
        "are missing:\n"
        f"{missing_cluster_page_files[:20]}"
    )


# -----------------------------------------------------
# Print summary
# -----------------------------------------------------

crops_with_iiif_pages = int(
    (
        corpus_metadata[
            "iiif_page_url"
        ]
        != ""
    ).sum()
)

clusters_with_iiif_pages = (
    corpus_metadata.loc[
        corpus_metadata[
            "iiif_page_url"
        ]
        != "",
        "cluster_id",
    ]
    .nunique()
)

print("=" * 75)
print("HTML FRONTEND WITH LIGHTBOX CREATED")
print("=" * 75)

print(
    "Frontend directory:",
    HTML_FRONTEND_DIR,
)

print()

print(
    "Cluster overview cards:",
    f"{len(cluster_summary):,}",
)

print(
    "Individual cluster pages:",
    f"{len(generated_cluster_pages):,}",
)

print(
    "Crops available in lightbox:",
    f"{len(corpus_metadata):,}",
)

print(
    "Crops with page-level IIIF links:",
    f"{crops_with_iiif_pages:,}",
)

print(
    "Clusters containing page-level IIIF links:",
    f"{clusters_with_iiif_pages:,}",
)

print()

print("Main index:")
print(HTML_INDEX_PATH)

print()

print(
    "Click a cluster card to open its cluster page."
)

print(
    "Click a crop to open the lightbox."
)

print(
    "Use the arrows or keyboard arrow keys to navigate."
)

print(
    "Double-click the enlarged crop to open its IIIF page."
)

print("=" * 75)